# <font color = yellow><b> Implementation of FOMC text RAG system as seperate agent for relevant text retrieval to answer user query.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
## Loading updated FOMC dataset generated during EDA(data_load_fomc.ipynb).

import contextlib
import io
from google.colab import drive

with contextlib.redirect_stdout(io.StringIO()):
    drive.mount('/content/drive')

path = "/content/FOMC_updated_dataset.xlsx"

fomc_df = pd.read_excel(path)


In [3]:
fomc_df.head()

,Date,Release Date,Type,Text,Release Year,Date Difference
0,2025-12-10,2025-12-30,Minute,Minutes of the Federal Open Market Committee\n...,2025,20
1,2025-12-10,2025-12-10,Statement,Available indicators suggest that economic act...,2025,0
2,2025-10-29,2025-10-29,Statement,Available indicators suggest that economic act...,2025,0
3,2025-10-29,2025-11-19,Minute,Minutes of the Federal Open Market Committee\n...,2025,21
4,2025-09-17,2025-09-17,Statement,Recent indicators suggest that growth of econo...,2025,0


#### FOMC dataset consists of six columns : Date, Release Date, Type, Text, Release Year, Date Difference.

#### Step 2: Transfomration of dataset.

In [4]:
## Copy FOMC uploaded dataset to different dataset for preprocessing.
fomc_new_df = fomc_df.copy()

In [5]:
## Keep 3required columns release date, type, text and release year
fomc_new_df = fomc_new_df[['Date','Release Date', 'Type', 'Text']]
fomc_new_df.head()


,Date,Release Date,Type,Text
0,2025-12-10,2025-12-30,Minute,Minutes of the Federal Open Market Committee\n...
1,2025-12-10,2025-12-10,Statement,Available indicators suggest that economic act...
2,2025-10-29,2025-10-29,Statement,Available indicators suggest that economic act...
3,2025-10-29,2025-11-19,Minute,Minutes of the Federal Open Market Committee\n...
4,2025-09-17,2025-09-17,Statement,Recent indicators suggest that growth of econo...


In [6]:
fomc_new_df.isna().sum()  ## To cehck if any null values present

,0
Date,0
Release Date,0
Type,0
Text,0


#### There are no missing records.

In [7]:
## Modify release date type to datetime if it is not datetime.

fomc_new_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 458 entries, 0 to 457
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          458 non-null    datetime64[ns]
 1   Release Date  458 non-null    datetime64[ns]
 2   Type          458 non-null    object        
 3   Text          458 non-null    object        
dtypes: datetime64[ns](2), object(2)
memory usage: 14.4+ KB


#### Release Date column is already a datetime type

In [8]:
## Add a new column doc_id with format M_actualmeetimgdate(yymmdd) for Minutes and S_actualmeetingdate(yymmdd) for Statements.
## doc_id is used to identify individual FOMC text.


for i in fomc_new_df.index:
  if fomc_new_df.iloc[i]['Type']=='Statement':
    fomc_new_df.loc[i,'doc_id'] = 'S_'+fomc_new_df.iloc[i]['Date'].strftime("%y%m%d")
  else:
    fomc_new_df.loc[i,'doc_id'] = 'M_'+fomc_new_df.iloc[i]['Date'].strftime("%y%m%d")



In [9]:
fomc_new_df.head(10)

,Date,Release Date,Type,Text,doc_id
0,2025-12-10,2025-12-30,Minute,Minutes of the Federal Open Market Committee\n...,M_251210
1,2025-12-10,2025-12-10,Statement,Available indicators suggest that economic act...,S_251210
2,2025-10-29,2025-10-29,Statement,Available indicators suggest that economic act...,S_251029
3,2025-10-29,2025-11-19,Minute,Minutes of the Federal Open Market Committee\n...,M_251029
4,2025-09-17,2025-09-17,Statement,Recent indicators suggest that growth of econo...,S_250917
5,2025-09-17,2025-10-08,Minute,Minutes of the Federal Open Market Committee\n...,M_250917
6,2025-07-30,2025-07-30,Statement,Although swings in net exports continue to aff...,S_250730
7,2025-07-30,2025-08-20,Minute,Minutes of the Federal Open Market Committee\n...,M_250730
8,2025-06-18,2025-06-18,Statement,Although swings in net exports have affected t...,S_250618
9,2025-06-18,2025-07-09,Minute,Minutes of the Federal Open Market Committee\n...,M_250618


### Doc_id indexes individual documents based on its type and release date for easy understanding of document indexing.

In [10]:
## Clean categorical column (type) by stripping spaces left and right.
fomc_new_df['Type'] = fomc_new_df['Type'].str.strip()


### Step 3: Cleaning of Text by removing header and short sentences not relevant for FOMC RAG implementation.

In [11]:
## Create adiitional column 'Text_Clean' and copy 'Text' column for cleaning of text.
fomc_new_df['Text_Clean'] = fomc_new_df['Text']


In [12]:
## Checking few sample Minutes.
from IPython.display import display
display(fomc_new_df[fomc_new_df.Type=='Minute'].Text.iloc[1])

'Minutes of the Federal Open Market Committee\n                                                \n                    \n                    \n                    \n                    October 28â\x80\x9329, 2025\nA joint meeting of the Federal Open Market Committee and the Board of Governors of the Federal Reserve System was held in the offices of the Board of Governors on Tuesday, October 28, 2025, at 9:00 a.m. and continued on Wednesday, October 29, 2025, at 9:00 a.m.1\n\nDevelopments in Financial Markets and Open Market Operations\nThe manager turned first to an overview of broad market developments during the intermeeting period. Market participants left their macroeconomic outlooks little changed, and they appeared to continue to interpret data made available over the period as consistent with a resilient economy. In line with the stable outlook, investors\' expectations for the path of the policy rate, whether market based or survey based, were virtually unchanged over the period.

In [13]:
txt=fomc_new_df[fomc_new_df.Type=='Minute'].Text.iloc[90].split('\n')
txt

['Minutes of the Federal Open Market Committee',
 '                                                ',
 '                    ',
 '                    ',
 '                    ',
 '                    December 16-17, 2014',
 '',
 'A meeting of the Federal Open Market Committee was held in the offices of the Board of Governors of the Federal Reserve System in Washington, D.C., on Tuesday, December 16, 2014, at 1:00 p.m. and continued on Wednesday, December 17, 2014, at 9:00 a.m.',
 '',
 'PRESENT:',
 '',
 'Janet L. Yellen, Chair',
 'William C. Dudley, Vice Chairman',
 'Lael Brainard',
 'Stanley Fischer',
 'Richard W. Fisher',
 'Narayana Kocherlakota',
 'Loretta J. Mester',
 'Charles I. Plosser',
 'Jerome H. Powell',
 'Daniel K. Tarullo',
 '',
 'Christine Cumming, Charles L. Evans, Jeffrey M. Lacker, Dennis P. Lockhart, and John C. Williams, Alternate Members of the Federal Open Market Committee',
 '',
 'James Bullard, Esther L. George, and Eric Rosengren, Presidents of the Federal Reserve 

### Minutes startes with header sentences and details of the meeting location and timestamp. It also contains board members name. These details do not add any meaning for analysis and hence need to be removed as part of text cleaning.

In [14]:
## To check first few sentences in Minutes
txt=[]
minutes_df=fomc_new_df[fomc_new_df.Type=='Minute']
for m in range(len(minutes_df)):
  txt.append(minutes_df.Text.iloc[m].split('\n'))  ## Splitting


## Observation:<br>
1. In minutes there are many sentences including header of meeting, participants name and meeting location are present which does not add any meaning to the minutes content. These needs to be removed.
2. Below are few samples sentences<br>
   'Minutes of the Federal Open Market Committee'<br>
   'A meeting of the Federal Open Market Committee was held in the offices....'<br>
   'PRESENT: Mr. Bernanke, ChairmanMr. Dudley, Vice ChairmanMs. DukeMr. EvansMr. KohnMr. LackerMr. Lockhart Mr. WarshMs. Yellen'<br>
   'Ms. Cumming, Mr. Fisher, Ms. Pianalto, and Messrs. Plosser and Stern, Alternate Members of the Federal Open Market Committee'


### These unneccessary sentences can be removed by direct delete and by using reular expression.

In [15]:
## Checking first sentrence of all minutes
for lst in txt:
  print(lst[0])

Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of the Federal Open Market Committee
Minutes of

#### Delete first default header of minutes.

In [16]:
## To delete first sentence of minutes as these are header of meetings.

txt_new = [lst[1:] for lst in txt]

## Re-checking post deletion of header
for lst in txt_new:
  print(lst[0])

#### To remove white spaces and empty lines

In [17]:
## Removing white spaces and empty lines
for i in range(len(txt_new)):
  txt_new[i] = [sent.strip() for sent in txt_new[i] if sent.strip()]
## Re-checking post removal of empty lines and spaces
for lst in txt_new:
  print(lst[1])


A joint meeting of the Federal Open Market Committee and the Board of Governors of the Federal Reserve System was held in the offices of the Board of Governors on Tuesday, December 9, 2025, at 9:00 a.m. and continued on Wednesday, December 10, 2025, at 9:00 a.m.1
A joint meeting of the Federal Open Market Committee and the Board of Governors of the Federal Reserve System was held in the offices of the Board of Governors on Tuesday, October 28, 2025, at 9:00 a.m. and continued on Wednesday, October 29, 2025, at 9:00 a.m.1
A joint meeting of the Federal Open Market Committee and the Board of Governors of the Federal Reserve System was held in the offices of the Board of Governors on Tuesday, September 16, 2025, at 10:30 a.m. and continued on Wednesday, September 17, 2025, at 9:00 a.m.1
A joint meeting of the Federal Open Market Committee and the Board of Governors of the Federal Reserve System was held in the offices of the Board of Governors on Tuesday, July 29, 2025, at 9:00 a.m. and c

### Using regular expression to remove sentences starting with 'A joint meeting of the Federal Open Market Committee and the Board of Governors'

In [18]:
## Checking few samples of sentences starting with 'A joint meeting of the Federal Open Market Committee and the Board of Governors'
cnt=0
for i in range(len(txt_new)):
  for sent in txt_new[i]:
    if sent.lower().startswith('a joint meeting of the federal open market committee and the board of governors'):
      cnt+=1
      print(sent)
print(cnt)

A joint meeting of the Federal Open Market Committee and the Board of Governors of the Federal Reserve System was held in the offices of the Board of Governors on Tuesday, December 9, 2025, at 9:00 a.m. and continued on Wednesday, December 10, 2025, at 9:00 a.m.1
A joint meeting of the Federal Open Market Committee and the Board of Governors of the Federal Reserve System was held in the offices of the Board of Governors on Tuesday, October 28, 2025, at 9:00 a.m. and continued on Wednesday, October 29, 2025, at 9:00 a.m.1
A joint meeting of the Federal Open Market Committee and the Board of Governors of the Federal Reserve System was held in the offices of the Board of Governors on Tuesday, September 16, 2025, at 10:30 a.m. and continued on Wednesday, September 17, 2025, at 9:00 a.m.1
A joint meeting of the Federal Open Market Committee and the Board of Governors of the Federal Reserve System was held in the offices of the Board of Governors on Tuesday, July 29, 2025, at 9:00 a.m. and c

In [19]:
## Removing all the baove sentences starting with 'A joint meeting of the Federal Open Market Committee and the Board of Governors'

for i in range(len(txt_new)):
    txt_new[i] = [sent for sent in txt_new[i] if not sent.lower().startswith('a joint meeting of the federal open market committee and the board of governors')]

## Verify sentence post removal
cnt=0
for i in range(len(txt_new)):
  for sent in txt_new[i]:
    if sent.lower().startswith('a joint meeting of the federal open market committee and the board of governors'):
      cnt+=1
      print(sent)
print(cnt)

0


#### Using regular expression to remove sentences starting with 'A meeting of the Federal Open Market Committee was held in the offices of the Board of Governors'

In [20]:
## Removing all the sentences starting with 'A meeting of the Federal Open Market Committee was held in the offices of the Board of Governors'

for i in range(len(txt_new)):
    txt_new[i] = [sent for sent in txt_new[i] if not sent.lower().startswith('a meeting of the federal open market committee was held in the offices of the board of governors')]

## Verify sentence post removal
cnt=0
for i in range(len(txt_new)):
  for sent in txt_new[i]:
    if sent.lower().startswith('a meeting of the federal open market committee was held in the offices of the board of governors'):
      cnt+=1
      print(sent)
print(cnt)

0


#### Using regular expression to remove senetnces with strings 'A meeting of the Federal Open Market Committee was held in the offices of the Board of Governors' post meeting date.

In [21]:
## Removing all the sentences having string 'A meeting of the Federal Open Market Committee was held in the offices of the Board of Governors'
## post meeting date.

string = 'a meeting of the federal open market committee was held in the offices of the board of governors'

for i in range(len(txt_new)):
    txt_new[i] = [sent for sent in txt_new[i] if string not in sent.lower()]


cnt=0
for i in range(len(txt_new)):
  for sent in txt_new[i]:
    if string in sent.lower():
      cnt+=1
      print(sent)
print(cnt)

0


In [22]:
## To check the paragraphs that starts with 'PRESENT' or 'ATTENDANCE' or sentences with participants name.

import re
pattern=re.compile(r"^(ms|mr|messrs|chair|vice chair|governer|secretary|deputy secretary|assistant secretary|general counsel|deputy general counsel|economist|manager|deputy manager)\b")
cnt=0
sent_remove=[]
for i in range(len(txt_new)):
  for sent in txt_new[i]:
    if sent.lower().startswith(('present','attendance')):
      cnt+=1
      # print(sent)
      sent_remove.append(sent)
    elif bool(pattern.search(sent.lower())):
      ## To retain sentences that have Secretary's note and Mr. Lacker actions as these are relevant content.
      if len(sent.split())<50 and not sent.lower().startswith(("secretary's note","mr. lacker dissented because")):
        cnt+=1
        # print(sent)
        sent_remove.append(sent)

print('\n'.join(sent_remove[700:900]))  ## Sample check to verfiy that relevant content are not removed.
print(cnt)


Mr. Struckmeyer, Deputy Staff Director, Office of Staff Director for Management, Board of Governors
Mr. Gagnon, Visiting Associate Director, Division of Monetary Affairs, Board of Governors
Messrs. Reifschneider and Wascher, Associate Directors, Division of Research and Statistics, Board of Governors
Mr. Oliner, Senior Adviser, Division of Research and Statistics, Board of Governors
Mr. Small, Project Manager, Division of Monetary Affairs, Board of Governors
Mr. Luecke, Section Chief, Division of Monetary Affairs, Board of Governors
Mr. Carlson, Economist, Division of Monetary Affairs, Board of Governors
Ms. Low, Open Market Secretariat Specialist, Division of Monetary Affairs, Board of Governors
Mr. Moore, First Vice President, Federal Reserve Bank of San Francisco
Mr. Judd, Executive Vice President, Federal Reserve Bank of San Francisco
Mr. Altig, Ms. Baum, Messrs. Rasche, Schweitzer, Sellon, and Tootell, Senior Vice Presidents, Federal Reserve Banks of Atlanta, New York, St. Louis, 

#### Using regular expression to remove participants name from Minutes and other irrelevant content from Minutes. There are total 3696 such sentences that needs to be removed.

In [23]:
## To remove the paragraphs that starts with 'PRESENT' or 'ATTENDANCE' or sentences with participants name.

pattern=re.compile(r"^(ms|mr|messrs|chair|vice chair|governer|secretary|deputy secretary|assistant secretary|general counsel|deputy general counsel|economist|manager|deputy manager)\b")
txt_new_final = [None] * len(txt_new)
for i in range(len(txt_new)):
  cleaned_txt=[]
  for sent in txt_new[i]:
    if sent.lower().startswith(('present','attendance')):
      continue
    elif bool(pattern.search(sent.lower())):
      if len(sent.split())<50 and not sent.lower().startswith(("secretary's note","mr. lacker dissented because")):
        continue
    cleaned_txt.append(sent)
  txt_new_final[i]=cleaned_txt

## To verify that cleaned text corpus does not have above mentioned irrelevant data.

cnt=0
sent_remove=[]
for i in range(len(txt_new_final)):
  for sent in txt_new_final[i]:
    if sent.lower().startswith(('present','attendance')):
      cnt+=1
      sent_remove.append(sent)
    elif bool(pattern.search(sent.lower())):
      ## To retain sentences that have Secretary's note and Mr. Lacker actions as these are relevant content.
      if len(sent.split())<50 and not sent.lower().startswith(("secretary's note","mr. lacker dissented because")):
        cnt+=1
        sent_remove.append(sent)
print(cnt)


0


In [24]:
## Minutes from July 2024 till Jan 2026 now contains cleaned content.
## However, documents previous to July 2024 still have participants name as they don't have initials and hence not removed in previous step.

ptrn1=re.compile(r"^([A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+)?)"
    r"(?:\s*,\s*(?:and\s+)?[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+)?)+")

cnt=0
sent_remove=[]
for i in range(len(txt_new_final)):
  for sent in txt_new_final[i]:
    if bool(ptrn1.search(sent)):
      cnt+=1
      sent_remove.append(sent)
print(cnt)

4407


#### There are total 4407 sentences that have participants name without initials which are to be removed using regex.

In [25]:
## To remove the paragraphs that have participants name.

ptrn1=re.compile(r"^([A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+)?)"
    r"(?:\s*,\s*(?:and\s+)?[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+)?)+")


txt_new_finalversion = [None] * len(txt_new_final)
for i in range(len(txt_new_final)):
  cleaned_txt=[]
  for sent in txt_new_final[i]:
    if bool(ptrn1.search(sent)):
      continue
    cleaned_txt.append(sent)
  txt_new_finalversion[i]=cleaned_txt

## To verify that cleaned text corpus does not have above mentioned irrelevant data.

cnt=0
sent_remove=[]
for i in range(len(txt_new_finalversion)):
  for sent in txt_new_finalversion[i]:
    if bool(ptrn1.search(sent)):
      cnt+=1
      sent_remove.append(sent)
print(cnt)

0


In [26]:
txt_new_finalversion[200] ## Randomly checking contents of minutes

['August 12, 2003',
 'By unanimous vote, Charles L. Evans and Brian F. Madigan were elected as Associate Economists of the Committee to serve until the election of their successors at the first regularly scheduled meeting of the Committee after December 31, 2003, with the understanding that in the event of the discontinuance of their official connection with the Board of Governors or with a Federal Reserve Bank, they would cease to have any official connection with the Federal Open Market Committee.',
 "The Manager of the System Open Market Account reported on recent developments in foreign exchange markets.  There were no open market operations in foreign currencies for the System's account in the period since the previous meeting.",
 'The Manager also reported on developments in domestic financial markets and on System open market transactions in government securities and securities issued or fully guaranteed by federal agencies during the period May 6, 2003, through June 24, 2003.  

### Now we have cleaned FOMC minutes which can be used for chunking and embeddings.

In [27]:
len(txt_new_finalversion)

238

In [28]:
txt_new_finalversion[0]

['December 9â\x80\x9310, 2025',
 'Developments in Financial Markets and Open Market Operations',
 "The manager turned first to an overview of broad market developments during the intermeeting period. Market participants did not materially change their macroeconomic outlooks and continued to interpret data made available over the intermeeting period as consistent with a resilient economy. Investors' expectations for the path of the policy rate, whether market based or survey based, were little changed, on net, over the period. Market participants and respondents to the Open Market Desk's Survey of Market Expectations (Desk survey) generally expected a 25 basis point reduction in the target range for the federal funds rate at the December FOMC meeting, and the modal outlook from the survey as well as from options pricing implied two additional rate cuts next year.",
 'The manager turned next to developments in Treasury markets and market-based measures of inflation compensation. Treasury

In [29]:
## Updating the cleaned Minute documents in the dataframe.

ind=fomc_new_df[fomc_new_df['Type']=='Minute'].index.to_list()

m=0
for i in ind:
  fomc_new_df.loc[i,'Text_Clean']="\n".join(txt_new_finalversion[m])
  m+=1



#### Column Text_Clean in dataframe now contains clean relevant content in Minutes.

## Chunking of FOMC documents

In [30]:
## FOMC statements and minutes consists of section header and contents and there are no tables or chart.
## Thus chunking based on section or paragraph level will be suitable for FOMC documents.

def minutes_chunks(index):
  section_header='Opening_Section'
  chunks=[]
  content=[]
  lines=fomc_new_df.iloc[index].Text_Clean.split('\n')
  for sent in lines:
    if not sent.endswith('.') and len(sent.split(' '))<25 and sum(1 for w in sent.split() if w[0].isupper())/len(sent.split())>0.7:
      if content:
        chunks.append({'section_header':section_header,'section_content':'\n'.join(content),'token_length':len(('\n'.join(content)).split(' '))})
        section_header=sent
        content=[]
      else:
        content.append(sent)
    else:
      content.append(sent)
  if content:
    chunks.append({'section_header':section_header,'section_content':'\n'.join(content),'token_length':len(('\n'.join(content)).split(' '))})
  return chunks



In [31]:
# section_chunk = list(minutes_chunks(0)[0].values())
# list(section_chunk[0].values())

section_chunk = minutes_chunks(0)
# list(section_chunk[0].values())[0]
for i in range(len(section_chunk)):
  print(list(section_chunk[i].values())[0])
  # print(lst)

Opening_Section
Developments in Financial Markets and Open Market Operations
Special Topic: Balance Sheet Issues
Staff Economic Outlook


In [32]:
fomc_new_df.head()

,Date,Release Date,Type,Text,doc_id,Text_Clean
0,2025-12-10,2025-12-30,Minute,Minutes of the Federal Open Market Committee\n...,M_251210,"December 9â10, 2025\nDevelopments in Financi..."
1,2025-12-10,2025-12-10,Statement,Available indicators suggest that economic act...,S_251210,Available indicators suggest that economic act...
2,2025-10-29,2025-10-29,Statement,Available indicators suggest that economic act...,S_251029,Available indicators suggest that economic act...
3,2025-10-29,2025-11-19,Minute,Minutes of the Federal Open Market Committee\n...,M_251029,"October 28â29, 2025\nDevelopments in Financi..."
4,2025-09-17,2025-09-17,Statement,Recent indicators suggest that growth of econo...,S_250917,Recent indicators suggest that growth of econo...


In [33]:
## Call chunking function for each minutes and store section chunks in new dataframe. For Statements, entire document is stored as one chunk as it has less words.

fomc_chunk_dict=[]
for idx in range(fomc_new_df.shape[0]):
  if fomc_new_df.iloc[idx].Type=='Minute':
    section_chunk = minutes_chunks(idx)
    for i in range(len(section_chunk)):
      fomc_chunk_dict.append({
          "doc_id":fomc_new_df.iloc[idx].doc_id,
          "meeting_date":fomc_new_df.iloc[idx].Date,
          "release_date":fomc_new_df.iloc[idx]['Release Date'],
          "doc_type":fomc_new_df.iloc[idx].Type,
          "chunk_id":f"{fomc_new_df.iloc[idx].doc_id}_chunk_{i+1}",
          "section_header":(list(section_chunk[i].values())[0]),
          "chunked_text":(list(section_chunk[i].values())[1]),
          "word_count":(list(section_chunk[i].values())[2])
        })
  else:
    fomc_chunk_dict.append({
          "doc_id":fomc_new_df.iloc[idx].doc_id,
          "meeting_date":fomc_new_df.iloc[idx].Date,
          "release_date":fomc_new_df.iloc[idx]['Release Date'],
          "doc_type":fomc_new_df.iloc[idx].Type,
          "chunk_id":f"{fomc_new_df.iloc[idx].doc_id}_chunk_1",
          "section_header":"Full Statement Doc",
          "chunked_text":fomc_new_df.iloc[idx].Text,
          "word_count":len(fomc_new_df.iloc[idx].Text.split())
        })



fomc_chunk_df=pd.DataFrame(fomc_chunk_dict)

In [34]:
fomc_chunk_df.shape  ## There are total 2354 chunks created of FOMC documents from 2000 to 2026.

(2354, 8)

In [35]:
fomc_chunk_df.head(20)

,doc_id,meeting_date,release_date,doc_type,chunk_id,section_header,chunked_text,word_count
0,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_1,Opening_Section,"December 9â10, 2025",3
1,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_2,Developments in Financial Markets and Open Mar...,The manager turned first to an overview of bro...,1056
2,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_3,Special Topic: Balance Sheet Issues,Participants discussed developments in money m...,2001
3,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_4,Staff Economic Outlook,Relative to the forecast prepared for the Octo...,1675
4,S_251210,2025-12-10,2025-12-10,Statement,S_251210_chunk_1,Full Statement Doc,Available indicators suggest that economic act...,402
5,S_251029,2025-10-29,2025-10-29,Statement,S_251029_chunk_1,Full Statement Doc,Available indicators suggest that economic act...,382
6,M_251029,2025-10-29,2025-11-19,Minute,M_251029_chunk_1,Opening_Section,"October 28â29, 2025",3
7,M_251029,2025-10-29,2025-11-19,Minute,M_251029_chunk_2,Developments in Financial Markets and Open Mar...,The manager turned first to an overview of bro...,904
8,M_251029,2025-10-29,2025-11-19,Minute,M_251029_chunk_3,Special Topic: The Standing Repo Facility,"The staff provided background on the SRF, incl...",193
9,M_251029,2025-10-29,2025-11-19,Minute,M_251029_chunk_4,Special Topic: Balance Sheet Issues,"The FOMC's ""Plans for Reducing the Size of the...",2169


In [36]:
fomc_chunk_df[fomc_chunk_df['word_count']>1000]

,doc_id,meeting_date,release_date,doc_type,chunk_id,section_header,chunked_text,word_count
1,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_2,Developments in Financial Markets and Open Mar...,The manager turned first to an overview of bro...,1056
2,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_3,Special Topic: Balance Sheet Issues,Participants discussed developments in money m...,2001
3,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_4,Staff Economic Outlook,Relative to the forecast prepared for the Octo...,1675
9,M_251029,2025-10-29,2025-11-19,Minute,M_251029_chunk_4,Special Topic: Balance Sheet Issues,"The FOMC's ""Plans for Reducing the Size of the...",2169
10,M_251029,2025-10-29,2025-11-19,Minute,M_251029_chunk_5,Staff Economic Outlook,Relative to the forecast prepared for the Sept...,1429
...,...,...,...,...,...,...,...,...
2328,M_000822,2000-08-22,2000-10-05,Minute,M_000822_chunk_2,Members of the Federal Open Market Committee,Monetary Affairs and Research and Statistics r...,3840
2332,M_000628,2000-06-28,2000-08-24,Minute,M_000628_chunk_3,"and Simpson, Associate Economists",Monetary Affairs and Research and Statistics r...,3824
2336,M_000516,2000-05-16,2000-06-29,Minute,M_000516_chunk_1,Opening_Section,"May 16, 2000\nThe Manager of the System Open M...",3550
2339,M_000321,2000-03-21,2000-05-18,Minute,M_000321_chunk_1,Opening_Section,"March 21, 2000\nThe Manager of the System Open...",4045


## Need to split chunks that have word length greater than 500 into '500' equal word length chunks for better performance.

In [37]:
def chunk_split_greater_than_1000(text):
  splt=[]
  for i in range(0,len(text.split()),500):
    splt.append(' '.join(text.split()[i:i+500]))
  return splt


fomc_chunk_dict_new=[]

for idx in range(fomc_chunk_df.shape[0]):
  if fomc_chunk_df.iloc[idx].word_count>500:
    split_text= chunk_split_greater_than_1000(fomc_chunk_df.iloc[idx].chunked_text)
    for i in range(len(split_text)):
      fomc_chunk_dict_new.append({
          "doc_id":fomc_chunk_df.iloc[idx].doc_id,
          "meeting_date":fomc_chunk_df.iloc[idx].meeting_date,
          "release_date":fomc_chunk_df.iloc[idx].release_date,
          "doc_type":fomc_chunk_df.iloc[idx].doc_type,
          "chunk_id":f"{fomc_chunk_df.iloc[idx].chunk_id}_{chr(96+i+1)}",
          "section_header":fomc_chunk_df.iloc[idx].section_header,
          "chunked_text":split_text[i],
          "word_count":len(split_text[i].split())
        })
  else:
    fomc_chunk_dict_new.append({
          "doc_id":fomc_chunk_df.iloc[idx].doc_id,
          "meeting_date":fomc_chunk_df.iloc[idx].meeting_date,
          "release_date":fomc_chunk_df.iloc[idx].release_date,
          "doc_type":fomc_chunk_df.iloc[idx].doc_type,
          "chunk_id":fomc_chunk_df.iloc[idx].chunk_id,
          "section_header":fomc_chunk_df.iloc[idx].section_header,
          "chunked_text":fomc_chunk_df.iloc[idx].chunked_text,
          "word_count":fomc_chunk_df.iloc[idx].word_count
        })

fomc_chunk_new_df=pd.DataFrame(fomc_chunk_dict_new)

In [38]:
fomc_chunk_new_df.head(20)

,doc_id,meeting_date,release_date,doc_type,chunk_id,section_header,chunked_text,word_count
0,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_1,Opening_Section,"December 9â10, 2025",3
1,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_2_a,Developments in Financial Markets and Open Mar...,The manager turned first to an overview of bro...,500
2,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_2_b,Developments in Financial Markets and Open Mar...,previous episode of balance sheet runoff. Cons...,500
3,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_2_c,Developments in Financial Markets and Open Mar...,standing repo operations for monetary policy i...,64
4,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_3_a,Special Topic: Balance Sheet Issues,Participants discussed developments in money m...,500
5,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_3_b,Special Topic: Balance Sheet Issues,observed that triparty repo rates had been ave...,500
6,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_3_c,Special Topic: Balance Sheet Issues,the pace of payroll employment increases had s...,500
7,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_3_d,Special Topic: Balance Sheet Issues,"and the Bank of Mexico, with most others leavi...",500
8,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_3_e,Special Topic: Balance Sheet Issues,rates had moved sideways since the beginning o...,21
9,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_4_a,Staff Economic Outlook,Relative to the forecast prepared for the Octo...,500


In [39]:
fomc_chunk_new_df.shape

(4060, 8)

### Now we have total 4060 chunks of less than or equal to 500 word length.

### To remove chunks with word length less than 10 as these chunks are not relevant to original minute content , example board member name.

In [40]:
fomc_chunk_new_df[fomc_chunk_new_df['word_count']<10].sort_values(by='word_count',ascending=False)

,doc_id,meeting_date,release_date,doc_type,chunk_id,section_header,chunked_text,word_count
3921,M_010320,2001-03-20,2001-05-17,Minute,M_010320_chunk_3_b,Telephone Conferences,to reduce the discount rate by 50 basis points.,9
3910,M_010411,2001-04-11,2001-05-02,Minute,M_010411_chunk_3_b,Telephone Conferences,to reduce the discount rate by 50 basis points.,9
733,M_210728,2021-07-28,2021-08-18,Minute,M_210728_chunk_10,"Erin E. Ferris2 and Andrei Zlate, Principal Ec...","Isabel Cairó, Senior Economist, Division of Mo...",9
177,M_240612,2024-06-12,2024-07-03,Minute,M_240612_chunk_8,"Beth Anne Wilson, Economist","Julie Ann Remache, Deputy Manager, System Open...",9
3899,M_010418,2001-04-18,2001-05-09,Minute,M_010418_chunk_3_b,Telephone Conferences,to reduce the discount rate by 50 basis points.,9
...,...,...,...,...,...,...,...,...
1564,M_170201,2017-02-01,2017-02-22,Minute,M_170201_chunk_8,Janet L. Yellen,Chairman,1
1227,M_190130,2019-01-30,2019-02-20,Minute,M_190130_chunk_11,Jerome H. Powell,Chairman,1
2929,M_090116,2009-01-16,2009-02-06,Minute,M_090116_chunk_2,Ben S. Bernanke,Chairman,1
3439,M_050202,2005-02-02,2005-02-23,Minute,M_050202_chunk_4,Alan Greenspan,Chairman,1


In [41]:
fomc_chunk_new_df[fomc_chunk_new_df['word_count']<10].sort_values(by='word_count',ascending=False)

,doc_id,meeting_date,release_date,doc_type,chunk_id,section_header,chunked_text,word_count
3921,M_010320,2001-03-20,2001-05-17,Minute,M_010320_chunk_3_b,Telephone Conferences,to reduce the discount rate by 50 basis points.,9
3910,M_010411,2001-04-11,2001-05-02,Minute,M_010411_chunk_3_b,Telephone Conferences,to reduce the discount rate by 50 basis points.,9
733,M_210728,2021-07-28,2021-08-18,Minute,M_210728_chunk_10,"Erin E. Ferris2 and Andrei Zlate, Principal Ec...","Isabel Cairó, Senior Economist, Division of Mo...",9
177,M_240612,2024-06-12,2024-07-03,Minute,M_240612_chunk_8,"Beth Anne Wilson, Economist","Julie Ann Remache, Deputy Manager, System Open...",9
3899,M_010418,2001-04-18,2001-05-09,Minute,M_010418_chunk_3_b,Telephone Conferences,to reduce the discount rate by 50 basis points.,9
...,...,...,...,...,...,...,...,...
1564,M_170201,2017-02-01,2017-02-22,Minute,M_170201_chunk_8,Janet L. Yellen,Chairman,1
1227,M_190130,2019-01-30,2019-02-20,Minute,M_190130_chunk_11,Jerome H. Powell,Chairman,1
2929,M_090116,2009-01-16,2009-02-06,Minute,M_090116_chunk_2,Ben S. Bernanke,Chairman,1
3439,M_050202,2005-02-02,2005-02-23,Minute,M_050202_chunk_4,Alan Greenspan,Chairman,1


In [42]:
## To retain chunks with word length greater than 20.
fomc_chunk_new_df=fomc_chunk_new_df[fomc_chunk_new_df['word_count']>20]

In [43]:
## To remove chunk text having Footnotes.

fomc_chunk_new_df=fomc_chunk_new_df[~fomc_chunk_new_df["chunked_text"].str.contains(r"footnote", case=False, na=False)]

In [44]:
## To remove chunk text having Footnotes.

fomc_chunk_new_df=fomc_chunk_new_df[~fomc_chunk_new_df["chunked_text"].str.contains(r"footnote", case=False, na=False)]

In [45]:
## Creating new column metadata that defines the chunked document.

fomc_chunk_new_df['metadata']=fomc_chunk_new_df.apply(lambda x: {'doc_id':str(x['doc_id']),'meeting_date':str(x['meeting_date']),\
                                  'release_date':str(x['release_date']), 'doc_type':str(x['doc_type'].lower()), 'chunk_id':x['chunk_id'],\
                                                                 'meeting_year': int(pd.to_datetime(x['meeting_date']).year)},axis=1)


In [46]:
## Final check to ensure there are no missing values and duplicate entry of chunk id.

fomc_chunk_new_df.isna().sum()



,0
doc_id,0
meeting_date,0
release_date,0
doc_type,0
chunk_id,0
section_header,0
chunked_text,0
word_count,0
metadata,0


#### There are no missing values in any columns.

In [47]:
fomc_chunk_new_df[fomc_chunk_new_df["chunk_id"].duplicated(keep=False)] ## To check for any duplicate entries

,doc_id,meeting_date,release_date,doc_type,chunk_id,section_header,chunked_text,word_count,metadata


#### There are no duplicate entries of chunk_ids.

In [48]:
fomc_chunk_new_df.sort_values(by='word_count')  ## To check number of word counts

,doc_id,meeting_date,release_date,doc_type,chunk_id,section_header,chunked_text,word_count,metadata
8,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_3_e,Special Topic: Balance Sheet Issues,rates had moved sideways since the beginning o...,21,"{'doc_id': 'M_251210', 'meeting_date': '2025-1..."
343,M_230726,2023-07-26,2023-08-16,Minute,M_230726_chunk_6,Christopher J. Waller,"Kelly J. Dubbert and Kathleen O'Neill Paese, I...",21,"{'doc_id': 'M_230726', 'meeting_date': '2023-0..."
2545,S_110809,2011-08-09,2011-08-09,Statement,S_110809_chunk_1_b,Full Statement Doc,continue to describe economic conditions as li...,21,"{'doc_id': 'S_110809', 'meeting_date': '2011-0..."
319,M_230920,2023-09-20,2023-10-11,Minute,M_230920_chunk_6,Lorie K. Logan,Christopher J. Waller\nSusan M. Collins and Je...,21,"{'doc_id': 'M_230920', 'meeting_date': '2023-0..."
275,M_231213,2023-12-13,2024-01-03,Minute,M_231213_chunk_6,Lorie K. Logan,Christopher J. Waller\nSusan M. Collins and Je...,21,"{'doc_id': 'M_231213', 'meeting_date': '2023-1..."
...,...,...,...,...,...,...,...,...,...
39,M_250917,2025-09-17,2025-10-08,Minute,M_250917_chunk_4_a,Committee Policy Actions,In their discussions of monetary policy for th...,500,"{'doc_id': 'M_250917', 'meeting_date': '2025-0..."
44,M_250730,2025-07-30,2025-08-20,Minute,M_250730_chunk_3_a,Developments in Financial Markets and Open Mar...,The manager turned first to a review of financ...,500,"{'doc_id': 'M_250730', 'meeting_date': '2025-0..."
45,M_250730,2025-07-30,2025-08-20,Minute,M_250730_chunk_3_b,Developments in Financial Markets and Open Mar...,June quarter-end; and the rise in net Treasury...,500,"{'doc_id': 'M_250730', 'meeting_date': '2025-0..."
2,M_251210,2025-12-10,2025-12-30,Minute,M_251210_chunk_2_b,Developments in Financial Markets and Open Mar...,previous episode of balance sheet runoff. Cons...,500,"{'doc_id': 'M_251210', 'meeting_date': '2025-1..."


### Now we have final total 2788 chunks with maximum of 500 word length and minimum of 21 word length.

In [49]:
## Downoad of final version of fomc_chunk dataset
fomc_chunk_new_df.to_pickle("/content/fomc_chunk_new_df.pkl")

### Embedding of Chunked text using Finbert model and storing in Chromadb for retreival.

In [50]:
## Import chromadb for stroing of embedded chunks
!pip install -U -q chromadb
import chromadb
from chromadb.utils import embedding_functions

In [51]:
## To use Finbert Embedding model SentenceTransformerEmbeddingFunction.
# embed_model = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="ProsusAI/finbert")
embed_model=embedding_functions.SentenceTransformerEmbeddingFunction(model_name="mukaj/fin-mpnet-base")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [52]:
# # Using Persistent Client to store Chroma DB in local path drive.
# chroma_drive_path = '/content/drive/MyDrive/FOMC_ChromaDB_Data'
chroma_drive_path1 = '/content/FOMC_ChromaDB_Data_v2'

# ## instantiating PersistentClient of chromadb.
# client = chromadb.PersistentClient(path=chroma_drive_path)
client1 = chromadb.PersistentClient(path=chroma_drive_path1)

# # Defining collection set up in chroma to pass the embedding function for embedding document chunks.
# chroma_collection = client.get_or_create_collection(name='FOMC_Chroma_Client', embedding_function=embed_model)
chroma_collection1 = client1.get_or_create_collection(name='FOMC_Chroma_Client1', embedding_function=embed_model)


In [53]:
## Creating document, ids and metadatas for ingestion into collection.

ids = list(fomc_chunk_new_df['chunk_id'].astype(str))
docs = list(fomc_chunk_new_df['chunked_text'].astype(str))
metadata = list(fomc_chunk_new_df['metadata'])

In [54]:
## Adding ids, docs and metadata list into collection in batches for effective CPU usage and overload.

## using finbert model
# batch=100
# for i in range(0,len(docs),batch):
#   chroma_collection.add(documents= docs[i:i+batch], ids = ids[i:i+batch], metadatas = metadata[i:i+batch])

#using model mukaj/fin-mpnet-base
batch=100
for i in range(0,len(docs),batch):
 chroma_collection1.add(documents= docs[i:i+batch], ids = ids[i:i+batch], metadatas = metadata[i:i+batch])


#### It takes approximately 45 mins to create embeded chunks and indexing into database.


In [55]:
# ## Copying collection database from Google Drive to local path.

!pip install chromadb
import shutil, chromadb, os
from google.colab import drive
drive.mount('/content/drive')


# Copy complete folder to local Colab path
shutil.copytree(
        "/content/FOMC_ChromaDB_Data_v2", "/content/drive/MyDrive/FOMC_ChromaDB_Data_v2",
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/FOMC_ChromaDB_Data_v2'

In [56]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [57]:
import contextlib
import io
from google.colab import drive
!pip install chromadb
import shutil, chromadb, os

with contextlib.redirect_stdout(io.StringIO()):
    drive.mount('/content/drive')

# # To load chroma database from local path
chroma_drive_path='/content/drive/MyDrive/FOMC_ChromaDB_Data_v2'
# ## instantiating PersistentClient of chromadb.
client = chromadb.PersistentClient(path=chroma_drive_path)

# # Defining collection set up in chroma to pass the embedding function for embedding document chunks.
# chroma_collection = client.get_or_create_collection(name='FOMC_Chroma_Client', embedding_function=embed_model)
chroma_collection = client.get_or_create_collection(name='FOMC_Chroma_Client1', embedding_function=embed_model)

In [58]:
chromadb.__version__

'1.5.5'

In [59]:
chroma_collection.count() ## Check if all chunks are embeded and indexed

2788

In [60]:
## Verifying few embedding entries from collection using get method and including embeddings, documents and metadatas and uris in retrieval.

sample = chroma_collection.get(limit=5, include=["embeddings", "documents", "metadatas"])

for item in sample:
    print(sample['embeddings'].shape)

(5, 768)
(5, 768)
(5, 768)
(5, 768)
(5, 768)
(5, 768)
(5, 768)


In [61]:
sample

{'ids': ['M_251210_chunk_2_a',
  'M_251210_chunk_2_b',
  'M_251210_chunk_2_c',
  'M_251210_chunk_3_a',
  'M_251210_chunk_3_b'],
 'embeddings': array([[-0.00571093,  0.0500219 , -0.0321373 , ..., -0.02260594,
         -0.14643994,  0.0201575 ],
        [ 0.05549336, -0.01209756, -0.02762297, ..., -0.00354427,
         -0.0777295 , -0.00333712],
        [ 0.0608335 , -0.00136805, -0.00170315, ...,  0.03147561,
         -0.03645298,  0.03054258],
        [ 0.01924396, -0.00616677, -0.02502816, ..., -0.00638236,
         -0.08666972,  0.03668779],
        [ 0.06031217, -0.00597217, -0.02933399, ...,  0.01226316,
         -0.06693666,  0.01088226]]),
 'documents': ["The manager turned first to an overview of broad market developments during the intermeeting period. Market participants did not materially change their macroeconomic outlooks and continued to interpret data made available over the intermeeting period as consistent with a resilient economy. Investors' expectations for the path o

### Verify the retrieval layer

In [62]:
## ## Taking user query from user for document search and retrieval.
user_query = input("Please enter your query: ")

Please enter your query: views on inflation


In [63]:
## Defining Function for query search in chroma vector DB.

def query_engiene(user_query):

  ## Perform query on choromadb based on user query and retrieve top 15 relevant search from vector db.
  query_results = chroma_collection.query(query_texts=user_query,n_results=15)

  ## Creating Dataframe of query results.

  query_results_df=pd.DataFrame(columns=['Document','Metadatas','Distance'])
  query_results_df['Document']=query_results['documents'][0]
  query_results_df['Metadatas']=query_results['metadatas'][0]
  query_results_df['Distance']=query_results['distances'][0]
  query_results_df['Document_ID']=[i['doc_id'] for i in query_results_df['Metadatas']]
  query_results_df['Chunk_ID']=[i['chunk_id'] for i in query_results_df['Metadatas']]
  query_results_df['Meeting_Date']=[i['meeting_date'] for i in query_results_df['Metadatas']]
  ## Return sorted table on Distance.
  return(query_results_df.sort_values(by='Distance',ascending=True))

In [64]:
## Calling query_engiene function to obtain top 15 similar documents from chroma db.
top_15_similar_document=query_engiene(user_query)
top_15_similar_document.sort_values(by='Distance',ascending=True)

,Document,Metadatas,Distance,Document_ID,Chunk_ID,Meeting_Date
0,to be a bit less pressing than those on the do...,"{'doc_id': 'M_031209', 'doc_type': 'minute', '...",0.509848,M_031209,M_031209_chunk_3_g,2003-12-09 00:00:00
1,"pressures on labor resources. Nonetheless, the...","{'doc_id': 'M_061212', 'meeting_date': '2006-1...",0.545159,M_061212,M_061212_chunk_5_g,2006-12-12 00:00:00
2,indicated that they could support a proposal t...,"{'doc_type': 'minute', 'meeting_date': '2003-0...",0.545661,M_030506,M_030506_chunk_1_g,2003-05-06 00:00:00
3,their domestic operations from foreign competi...,"{'meeting_date': '2003-10-28 00:00:00', 'chunk...",0.548382,M_031028,M_031028_chunk_4_f,2003-10-28 00:00:00
4,warned about the macroeconomic consequences of...,"{'release_date': '2020-01-03 00:00:00', 'doc_i...",0.551839,M_191211,M_191211_chunk_10_c,2019-12-11 00:00:00
5,upside risks continued to cloud the outlook fo...,"{'doc_id': 'M_030916', 'meeting_date': '2003-0...",0.553441,M_030916,M_030916_chunk_4_g,2003-09-16 00:00:00
6,the expansion of potential GDP could be somewh...,"{'chunk_id': 'M_060920_chunk_1_f', 'doc_type':...",0.561726,M_060920,M_060920_chunk_1_f,2006-09-20 00:00:00
7,"will come into better balance over time, easin...","{'release_date': '2023-04-12 00:00:00', 'chunk...",0.567517,M_230322,M_230322_chunk_16_d,2023-03-22 00:00:00
8,The staff projection for economic activity was...,"{'meeting_date': '2025-01-29 00:00:00', 'doc_i...",0.569106,M_250129,M_250129_chunk_17_a,2025-01-29 00:00:00
9,of more widespread contagion. Given their anti...,"{'release_date': '2002-08-15 00:00:00', 'meeti...",0.569363,M_020626,M_020626_chunk_1_g,2002-06-26 00:00:00


### First few retrievals are relevant to the query context however there are many chunks that do not add any values as they contains mostly about the board member name who voted on actions.

### Second level re-ranking is required on top 10-20 retrieved chunks for better results. This is where we use cross-encider transmofer model Roberta for reranking.

In [65]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/stsb-roberta-large")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [66]:
## Function generated to return similarity or accuracy score of each document and query pair.

def rerank_top15_document(user_query,top_15_similar_document):

  ## Creating pair of document(top 10 documents obtianed from engiene query) and query list for re-ranking score.
  cross_encoder_sentence_query_pair = [[user_query,x] for x in top_15_similar_document.Document]

  ## Predicting scores of document and query pair.
  score=reranker.predict(cross_encoder_sentence_query_pair)

  ## Normalising score obtained above in range[0-1] by applying sigmoid function.
  score_norm = [round(float(1/(1+np.exp(-scr))),4) for scr in score]

  ## Updating score and normalised score in dataframe.
  top_15_similar_document['Encoder_Norm_score']=score_norm
  top_15_similar_document['Encoder_score']=score

  ## return sorted table by sorting Norm Score in Descending order.
  return(top_15_similar_document.sort_values(by='Encoder_Norm_score',ascending=False))


In [67]:
## Calling re-rank function to get normalised score of top 10 documents for further refine search result.
top_15_doc_reranked_score_df= rerank_top15_document(user_query,top_15_similar_document)

## Cheking score values.
print('Score from Cross-Encoder model :\n ',top_15_doc_reranked_score_df['Encoder_score'].tolist())
print('\nNormalized Score of score from Cross-Encoder model :\n ',top_15_doc_reranked_score_df['Encoder_Norm_score'].tolist())

Score from Cross-Encoder model :
  [0.6575573682785034, 0.5877796411514282, 0.5827152729034424, 0.5705476999282837, 0.5633572936058044, 0.5603565573692322, 0.5479186773300171, 0.526578426361084, 0.5221132040023804, 0.46715572476387024, 0.42996153235435486, 0.42996153235435486, 0.4280036687850952, 0.38446125388145447, 0.38446125388145447]

Normalized Score of score from Cross-Encoder model :
  [0.6587, 0.6429, 0.6417, 0.6389, 0.6372, 0.6365, 0.6337, 0.6287, 0.6276, 0.6147, 0.6059, 0.6059, 0.6054, 0.5949, 0.5949]


In [68]:
## Retrieving top 3 documents based on Encoder score.
top_15_doc_reranked_score_df.sort_values(by='Encoder_Norm_score',ascending=False)[0:3]

,Document,Metadatas,Distance,Document_ID,Chunk_ID,Meeting_Date,Encoder_Norm_score,Encoder_score
4,warned about the macroeconomic consequences of...,"{'release_date': '2020-01-03 00:00:00', 'doc_i...",0.551839,M_191211,M_191211_chunk_10_c,2019-12-11 00:00:00,0.6587,0.657557
0,to be a bit less pressing than those on the do...,"{'doc_id': 'M_031209', 'doc_type': 'minute', '...",0.509848,M_031209,M_031209_chunk_3_g,2003-12-09 00:00:00,0.6429,0.587780
1,"pressures on labor resources. Nonetheless, the...","{'doc_id': 'M_061212', 'meeting_date': '2006-1...",0.545159,M_061212,M_061212_chunk_5_g,2006-12-12 00:00:00,0.6417,0.582715


### Nowe we can see reranking model has performed better in terms of retrieveing most relevant content based on user query.

# <font color = yellow><b> Implementation of retrieval from FRED dataframes as seperate agent for relevant numerical data retrieval to answer user query that needs look up of numerical macroeconomic dataframe

### Step 1: Loading of saved copy of dataframes of GDP, Employment, Inflation and Fed Funds data from local drive

### First manaully load the datasets into colab drive.

In [69]:
from google.colab import files

gdp_fred_data = pd.read_csv("/content/gdp_fred_data.csv")
poppulation_data = pd.read_csv("/content/poppulation_data.csv")
inflation_data = pd.read_csv("/content/inflation_data.csv")
fed_fund_data = pd.read_csv("/content/fed_fund_data.csv")


In [70]:
## To verify all FRED dataframes are successfully loaded.

print('GDP data : Rows =',gdp_fred_data.shape[0])
print('First five records :\n\n',gdp_fred_data.head())
print('\n\nPoppulation data : Rows =',poppulation_data.shape[0])
print('First five records :\n\n',poppulation_data.head())
print('\n\nInflation data : Rows =',inflation_data.shape[0])
print('First five records :\n\n',inflation_data.head())
print('\n\nFed Fund data : Rows =',fed_fund_data.shape[0])
print('First five records :\n\n',fed_fund_data.head())


GDP data : Rows = 101
First five records :

          DATE      GDPC1  yoy_gdp_%
0  2000-10-01  14229.765       4.08
1  2001-01-01  14183.120       3.57
2  2001-04-01  14271.694       2.50
3  2001-07-01  14214.516       1.64
4  2001-10-01  14253.574       0.96


Poppulation data : Rows = 313
First five records :

          DATE  Civilian_Population_16+  Total_Labor_Force  \
0  2000-01-01                 211410.0           142267.0   
1  2000-02-01                 211576.0           142456.0   
2  2000-03-01                 211772.0           142434.0   
3  2000-04-01                 212018.0           142751.0   
4  2000-05-01                 212242.0           142388.0   

   Employed_Population  Participation_Rate_%  Employment_Rate_%  \
0             136559.0                 67.29              64.59   
1             136598.0                 67.33              64.56   
2             136701.0                 67.26              64.55   
3             137270.0                 67.33     

#### Step 2: To merge all FRED data into single dataframe by normalizing datasets. <br><br>
### Below are the data frequency of each fred dataset:<br>
gdp - quaterly (total 101 rows from 2000 to 2025)<br>poppulation - monthly (total 313 rows from 2000 to 2026)<br> inflation - daily excluding weekend (total 5797 rows from 2003 to 2026) <br>fed fund - monthly (total 301 rows from 2000 to 2025)


In [71]:
## Merging of FRED datasets to a master dataset for retreival.
## The master dataset will consists of columns Date, economic_indicator_desc, fred_api, frequency_type and rate_value_%.

## Creating new dataframe of GDP with these five columns.

gdp_fred_data_new=pd.DataFrame(columns=['Date','economic_indicator_desc','fred_api','frequency_type'])
gdp_fred_data_new['Date']=pd.to_datetime(gdp_fred_data['DATE'])
gdp_fred_data_new['economic_indicator_desc']='GDP Growth Rate'
gdp_fred_data_new['fred_api']='GDPC1'
gdp_fred_data_new['frequency_type']='Quarterly'
gdp_fred_data_new['rate_value_%']=gdp_fred_data['yoy_gdp_%']

In [72]:
gdp_fred_data_new.head()

,Date,economic_indicator_desc,fred_api,frequency_type,rate_value_%
0,2000-10-01,GDP Growth Rate,GDPC1,Quarterly,4.08
1,2001-01-01,GDP Growth Rate,GDPC1,Quarterly,3.57
2,2001-04-01,GDP Growth Rate,GDPC1,Quarterly,2.50
3,2001-07-01,GDP Growth Rate,GDPC1,Quarterly,1.64
4,2001-10-01,GDP Growth Rate,GDPC1,Quarterly,0.96


In [73]:
## Creating new dataframe of poppulation particiaption, employment and unemployment rate with these five columns -
## columns Date, economic_indicator_desc, fred_api, frequency_type and rate_value_%.

poppulation_participation_data_new=pd.DataFrame(columns=['Date','economic_indicator_desc','fred_api','frequency_type'])
poppulation_employment_data_new=pd.DataFrame(columns=['Date','economic_indicator_desc','fred_api','frequency_type'])
poppulation_unemployment_data_new=pd.DataFrame(columns=['Date','economic_indicator_desc','fred_api','frequency_type'])

## For participation rate
poppulation_participation_data_new['Date']=pd.to_datetime(poppulation_data['DATE'])
poppulation_participation_data_new['economic_indicator_desc']='Participation Rate'
poppulation_participation_data_new['fred_api']='CLF16OV'
poppulation_participation_data_new['frequency_type']='Monthly'
poppulation_participation_data_new['rate_value_%']=poppulation_data['Participation_Rate_%']
## For Employment Rate
poppulation_employment_data_new['Date']=pd.to_datetime(poppulation_data['DATE'])
poppulation_employment_data_new['economic_indicator_desc']='Employment Rate'
poppulation_employment_data_new['fred_api']='CE16OV'
poppulation_employment_data_new['frequency_type']='Monthly'
poppulation_employment_data_new['rate_value_%']=poppulation_data['Employment_Rate_%']
## For Un-employment Rate
poppulation_unemployment_data_new['Date']=pd.to_datetime(poppulation_data['DATE'])
poppulation_unemployment_data_new['economic_indicator_desc']='Un-employment Rate'
poppulation_unemployment_data_new['fred_api']='CLF16OV and CE16OV'
poppulation_unemployment_data_new['frequency_type']='Monthly'
poppulation_unemployment_data_new['rate_value_%']=poppulation_data['Unemployment_Rate_%']

In [74]:
## To recheck if correctly data loaded

poppulation_participation_data_new.head()

,Date,economic_indicator_desc,fred_api,frequency_type,rate_value_%
0,2000-01-01,Participation Rate,CLF16OV,Monthly,67.29
1,2000-02-01,Participation Rate,CLF16OV,Monthly,67.33
2,2000-03-01,Participation Rate,CLF16OV,Monthly,67.26
3,2000-04-01,Participation Rate,CLF16OV,Monthly,67.33
4,2000-05-01,Participation Rate,CLF16OV,Monthly,67.09


In [75]:
poppulation_employment_data_new.head()

,Date,economic_indicator_desc,fred_api,frequency_type,rate_value_%
0,2000-01-01,Employment Rate,CE16OV,Monthly,64.59
1,2000-02-01,Employment Rate,CE16OV,Monthly,64.56
2,2000-03-01,Employment Rate,CE16OV,Monthly,64.55
3,2000-04-01,Employment Rate,CE16OV,Monthly,64.74
4,2000-05-01,Employment Rate,CE16OV,Monthly,64.37


In [76]:
poppulation_unemployment_data_new.head()

,Date,economic_indicator_desc,fred_api,frequency_type,rate_value_%
0,2000-01-01,Un-employment Rate,CLF16OV and CE16OV,Monthly,4.01
1,2000-02-01,Un-employment Rate,CLF16OV and CE16OV,Monthly,4.11
2,2000-03-01,Un-employment Rate,CLF16OV and CE16OV,Monthly,4.03
3,2000-04-01,Un-employment Rate,CLF16OV and CE16OV,Monthly,3.84
4,2000-05-01,Un-employment Rate,CLF16OV and CE16OV,Monthly,4.04


In [77]:
## Creating new dataframe of inflation rate with these five columns - columns Date, economic_indicator_desc, fred_api, frequency_type and rate_value_%.

inflation_data_5year_new=pd.DataFrame(columns=['Date','economic_indicator_desc','fred_api','frequency_type'])
inflation_data_10year_new=pd.DataFrame(columns=['Date','economic_indicator_desc','fred_api','frequency_type'])

## For Inflation expected rate 5-year
inflation_data_5year_new['Date']=pd.to_datetime(inflation_data['DATE'])
inflation_data_5year_new['economic_indicator_desc']='Inflation Rate 5Year'
inflation_data_5year_new['fred_api']='T5YIE'
inflation_data_5year_new['frequency_type']='Daily'
inflation_data_5year_new['rate_value_%']=inflation_data['Breakdown_Inflation_5_year']

## For Inflation expected rate 10-year
inflation_data_10year_new['Date']=pd.to_datetime(inflation_data['DATE'])
inflation_data_10year_new['economic_indicator_desc']='Inflation Rate 10Year'
inflation_data_10year_new['fred_api']='T10YIE'
inflation_data_10year_new['frequency_type']='Daily'
inflation_data_10year_new['rate_value_%']=inflation_data['Breakdown_Inflation_10_year']

In [78]:
## To check if data correctly loaded

inflation_data_5year_new.head()

,Date,economic_indicator_desc,fred_api,frequency_type,rate_value_%
0,2003-01-02,Inflation Rate 5Year,T5YIE,Daily,1.30
1,2003-01-03,Inflation Rate 5Year,T5YIE,Daily,1.28
2,2003-01-06,Inflation Rate 5Year,T5YIE,Daily,1.31
3,2003-01-07,Inflation Rate 5Year,T5YIE,Daily,1.28
4,2003-01-08,Inflation Rate 5Year,T5YIE,Daily,1.33


In [79]:
inflation_data_10year_new.head()

,Date,economic_indicator_desc,fred_api,frequency_type,rate_value_%
0,2003-01-02,Inflation Rate 10Year,T10YIE,Daily,1.64
1,2003-01-03,Inflation Rate 10Year,T10YIE,Daily,1.62
2,2003-01-06,Inflation Rate 10Year,T10YIE,Daily,1.63
3,2003-01-07,Inflation Rate 10Year,T10YIE,Daily,1.62
4,2003-01-08,Inflation Rate 10Year,T10YIE,Daily,1.71


In [80]:
## Creating new dataframe of fed funds rate with these five columns - columns Date, economic_indicator_desc, fred_api, data_frequency and rate_value_%.

fed_fund_data_new=pd.DataFrame(columns=['Date','economic_indicator_desc','fred_api','frequency_type'])

## For Inflation expected rate 5-year
fed_fund_data_new['Date']=pd.to_datetime(fed_fund_data['DATE'])
fed_fund_data_new['economic_indicator_desc']='Federal Funds Rate'
fed_fund_data_new['fred_api']='FEDFUNDS'
fed_fund_data_new['frequency_type']='Monthly'
fed_fund_data_new['rate_value_%']=fed_fund_data['Federal_Funds_Rate_%']

In [81]:
## To check if data is correctly loaded

fed_fund_data_new.head()

,Date,economic_indicator_desc,fred_api,frequency_type,rate_value_%
0,2000-01-01,Federal Funds Rate,FEDFUNDS,Monthly,5.45
1,2000-02-01,Federal Funds Rate,FEDFUNDS,Monthly,5.73
2,2000-03-01,Federal Funds Rate,FEDFUNDS,Monthly,5.85
3,2000-04-01,Federal Funds Rate,FEDFUNDS,Monthly,6.02
4,2000-05-01,Federal Funds Rate,FEDFUNDS,Monthly,6.27


In [82]:
### To merge all the individual datasets created into one master FRED dataset.

fred_master_df = pd.concat([gdp_fred_data_new, poppulation_participation_data_new, poppulation_employment_data_new\
                            ,poppulation_unemployment_data_new,inflation_data_5year_new, inflation_data_10year_new, fed_fund_data_new]\
                           ,ignore_index=True)
fred_master_df

,Date,economic_indicator_desc,fred_api,frequency_type,rate_value_%
0,2000-10-01,GDP Growth Rate,GDPC1,Quarterly,4.08
1,2001-01-01,GDP Growth Rate,GDPC1,Quarterly,3.57
2,2001-04-01,GDP Growth Rate,GDPC1,Quarterly,2.50
3,2001-07-01,GDP Growth Rate,GDPC1,Quarterly,1.64
4,2001-10-01,GDP Growth Rate,GDPC1,Quarterly,0.96
...,...,...,...,...,...
12930,2024-09-01,Federal Funds Rate,FEDFUNDS,Monthly,5.13
12931,2024-10-01,Federal Funds Rate,FEDFUNDS,Monthly,4.83
12932,2024-11-01,Federal Funds Rate,FEDFUNDS,Monthly,4.64
12933,2024-12-01,Federal Funds Rate,FEDFUNDS,Monthly,4.48


## Total number of rows in FRED master dataset is 12935.

In [83]:
## To Verify and check there are no duplicates and data loaded are correct.

fred_master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12935 entries, 0 to 12934
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Date                     12935 non-null  datetime64[ns]
 1   economic_indicator_desc  12935 non-null  object        
 2   fred_api                 12935 non-null  object        
 3   frequency_type           12935 non-null  object        
 4   rate_value_%             12935 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 505.4+ KB


In [84]:
## To update the object datatype to string type.
fred_master_df["economic_indicator_desc"] = fred_master_df["economic_indicator_desc"].astype("string")
fred_master_df["fred_api"] = fred_master_df["fred_api"].astype("string")
fred_master_df["frequency_type"] = fred_master_df["frequency_type"].astype("string")


fred_master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12935 entries, 0 to 12934
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Date                     12935 non-null  datetime64[ns]
 1   economic_indicator_desc  12935 non-null  string        
 2   fred_api                 12935 non-null  string        
 3   frequency_type           12935 non-null  string        
 4   rate_value_%             12935 non-null  float64       
dtypes: datetime64[ns](1), float64(1), string(3)
memory usage: 505.4 KB


In [85]:
fred_master_df.groupby('economic_indicator_desc').count()  ## To verify the number of rows of each FRED indicator

,Date,fred_api,frequency_type,rate_value_%
economic_indicator_desc,,,,
Employment Rate,313,313,313,313
Federal Funds Rate,301,301,301,301
GDP Growth Rate,101,101,101,101
Inflation Rate 10Year,5797,5797,5797,5797
Inflation Rate 5Year,5797,5797,5797,5797
Participation Rate,313,313,313,313
Un-employment Rate,313,313,313,313


In [86]:
fred_master_df[fred_master_df.duplicated(subset=["Date", "economic_indicator_desc"])]

,Date,economic_indicator_desc,fred_api,frequency_type,rate_value_%


In [87]:
## Downloading the FRED master file using pickle to local system.
fred_master_df.to_pickle("/content/fred_master_df.pkl")
fomc_chunk_new_df.to_pickle("/content/fomc_chunk_new_df.pkl")


### There are no duplicate values and FRED data are correctly loaded into master dataframe which can be used for further retrieval.

# <font color = yellow><b># Genertate query parser using LLM to provide query metadata such as query type, economic indicator and time series for future retreival.

In [88]:
## install and import open ai

!pip install openai
import openai
from openai import OpenAI

In [89]:
## Creating system message and user message for prompt.

def query_parser_prompt_llm(user_query):
  system_message = """ You are an query parser assistant who is expertise in Macroeconomic analysis domain especially with FRED economic indcator and FOMC will help\
  to generate parameters from query in json format. """

  user_message = f"""
  You have knowledge in Economic Domain with FRED and FOMC and your job is to geneerate structured JSON from a given input query.
  You are given the query asked by user in variable <<user_query>> as below:
  <<user_query>> = '{user_query}'
  Your task is to list out the values of paramaters query_type, query_task_type, indicator, aggregation_method, start_dt, end_dt, chart_required from the input query provided in variable <<user_query>> and output these parameter value in JSON format.
  You should only use below values for query_type paramater:
  1. text
  2. numeric
  3. hybrid

  You should only use below values for <<query_task_type>> paramater:
  1. single
  2. timeseries
  3. compare_indicator
  4. compare_period
  5. summary_topic
  6. summary_fomc_statement
  7. summary_fomc_minute
  8. question_answer

   You should only use below values for <<indicator>> paramater:
  1. gdp_rate
  2. employment_rate
  3. unemployment_rate
  4. inflation_5year
  5. inflation_10year
  6. fed_fund_rate

  You should only use below values for <<aggregation_method>> paramater:
  1. Average
  2. Min
  3. Max
  4. Latest
  5. Oldest
  6. Median
  7. standard_deviation
  8. quantile

  You should only use below values for <<chart_required>> paramater:

  You must follow below rules to generate JSON output:
  1. If query can be answered only from FOMC minutes and statements, then 'query_type' parameter value should be 'text'.
  2. If query can be answered only from FRED numeric data, then 'query_type' parameter value should be 'numeric'.
  3. If query needs both FOMC text and FRED numeric, then 'query_type' parameter value should be 'hybrid'.
  4. If the query needs numeric FRED data and asks for a single value at a specific date, then 'query_task_type' should be 'single'.
  5. If the query needs numeric FRED data and ask about trend, pattern, change over timeperiod, then 'query_task_type' should be 'timeseries'.
  6. If the query needs numeric FRED data and asks comparisions between two or more indicators, then 'query_task_type' should be 'compare_indicator'.
  7. If the query needs numeric FRED data and and asks comparisons between different periods of an indicator, then 'query_task_type' should be 'compare_period'.
  8. If the query needs numeric FRED data and ask about separate two or more indicators and there is no word compare, then 'query_task_type' should be 'single'.
  9. If the query needs numeric FRED data and asks for chart or there is a need for chart plot, then 'chart_required' should be 'yes' else it should be 'no'.
  10. If the query needs numeric FRED data and is about gdp growth rate, then 'indicator' should be 'gdp_rate'.
  11. If the query needs numeric FRED data and is about employment growth rate, then 'indicator' should be 'employment_rate'.
  12. If the query needs numeric FRED data and is about unemployment growth rate, then 'indicator' should be 'unemployment_rate'.
  13. If the query needs numeric FRED data and is about the inflation rate in the short term, then 'indicator' should be 'inflation_5year'.
  14. If the query needs numeric FRED data and is about the inflation rate in the long term, then 'indicator' should be 'inflation_10year'.
  15. If the query needs numeric FRED data and is about the federal funds rate, then 'indicator' should be 'fed_fund_rate'.
  16. If the query needs numeric FRED data and asks for more than one indicator, then the 'indicator' parameter should have values in a list containing all the indicators.
  17. If the query needs text FOMC data and asks for a summary, brief, or overview of a topic or indicator, then 'query_task_type' should be 'summary_topic' and you need to identify the indicator and update the indicator in output dictionary item.
      Note that querry is categorised as 'summary' when the output should be a natural language generation  of retrieved data.
  18. If the query needs text FOMC data and asks about a summary, a brief, or an overview of FOMC minutes, then 'query_task_type' should be 'summary_fomc_minute'.
  19. If the query needs text FOMC data and asks about a summary and asks for a summary of FOMC statements, then 'query_task_type' should be 'summary_fomc_statement'.
  20. If the query needs text FOMC data and asks about information of indicators or questions of policy decision or question of facts, then 'query_task_type' should be 'question_answer' and 'indicator' can be empty.
  21. start_dt and end_dt should be in YYYY-MM-DD format.
  22. If start date and end date are not provided in the query, then you must take the last year of data, where start_dt should be "2025-01-01" and end_dt should be "2025-12-31".
  23. If the query needs numeric FRED data and 'query_task_type' is 'compare_period', then identify the start date and end date of both periods and store in start_dt and end_dt as list type.
  24. Do not return values other than those specified in the above list.
  25. For summary_fomc_statement and summary_fomc_minute query_task_type, if period range is not provided then keep start_dt and end_dt as empty.
  26. If the query needs numeric FRED data and asks for a single value at a specific date, then 'query_task_type' should be 'single' and <<aggregation_method>> parameter must be only one of the below:
      a. mean : If the query asks for the average or a general value of an indicator within given specific time period.
      b. min : If the query asks for the minimum or lowest value of an indicator within given specific time period.
      c. max : If the query asks for the maximum value of an indicator within a given specific time period.
      d. latest : If the query asks for the latest value or most recent values of an indicator within given specific time period.
      e. oldest : If the query asks for the oldest or earliest value of an indicator within given specific time period.
      f. median : If the query asks for the median or typical value of an indicator.
      g. standard_deviation : If the query asks for the standard deviation or volatility of an indicator.
      h. quantile : If the query asks about quantile or percentile values of an indicator.
      If nothing is provided in the query then you must use 'mean' as aggregation method.
      Do not generate any other aggregation_method other than mean, min, max, latest, oldest, median, standard_deviation and quantile.
  27. For query type 'numeric' and query_task_type other than 'single', then <<aggregation_method>> parameter must be empty.
  28. For 'text' query type the <<aggregation_method>> parameter must be empty.
  You should double verify that output generated is in below exact JSON format:
  {{"query_type": "value",
  "query_task_type": "value",
  "indicator": list of value,
  "aggregation_method": "value",
  "start_dt": date value or list of date values,
  "end_dt": date value or list of date values,
  "chart_required": "value"}}

  Note that query is provided in {user_query} and you must return a response in valid JSON format only.

  Refer below few shot examples to generate accurate JSON response of input user query:

  Example 1:
  user_query : Give some insights about inflation rate.
  response:
  {{"query_type": "text",
  "query_task_type": "question_answer",
  "indicator": [],
  "start_dt": "2025-01-01",
  "end_dt": "2025-12-31",
  "chart_required": "no"}}

  Example 2:
  user_query : Summarize FOMC meeting conducted in September 2023.
  response:
  {{"query_type": "text",
  "query_task_type": "summary_fomc_minute",
  "indicator": [],
  "start_dt": "2023-09-01",
  "end_dt": "2023-09-30",
  "chart_required": "no"}}

  Example 3:
  user_query :What was gdp rate in 2024.
  response:
  {{"query_type": "numeric",
  "query_task_type": "single",
  "indicator": [gdp_rate],
  "aggregation_method": "Average",
  "start_dt": "2024-01-01",
  "end_dt": "2024-12-31",
  "chart_required": "no"}}

  Example 4:
  User query: Show the trend of inflation in the short term from 2020 to 2023.
  response:
  {{"query_type": "numeric",
  "task_type": "timeseries",
  "indicators": ["inflation_5year"],
  "start_date": "2020-01-01",
  "end_date": "2023-12-31",
  "chart_required": "yes"}}

  Example 4:
  User query: What was the FED's view regarding unemployment?
  response:
  {{"query_type": "text",
  "task_type": "question_answer",
  "indicators": [],
  "start_date": "2025-01-01",
  "end_date": "2025-12-31",
  "chart_required": "no"}}

  Example 5:
  User query: Compare employment and unemployment rate.
  Output:
  {{"query_type": "numeric",
  "task_type": "compare_indicator",
  "indicators": ["employment_rate", "unemployment_rate"],
  "start_date": "2025-01-01",
  "end_date": "2025-12-31",
  "chart_required": "yes"}}

  """
  # Mount Google Drive and set OpenAI API key.
  from google.colab import drive

  with contextlib.redirect_stdout(io.StringIO()):
    drive.mount('/content/drive')

  # Set the API key
  filepath = "/content/drive/MyDrive/Agentic RAG Project/"
  with open(filepath + "OpenAI_API_Key.txt", "r") as f:
   openai.api_key = ' '.join(f.readlines())

  # Initialize client with API key
  client = OpenAI(api_key=openai.api_key)

  ## Call LLM model GPT-5.4 to generate query parser in JSON response. We arere using chat gpt model GPT-4o Realtime(gpt-4.5-preview).

  llm_response = openai.chat.completions.create( model="gpt-5.4",
                  messages=[{"role":"system", "content":system_message},
                             {"role":"user","content":user_message}], temperature=0
                  )
  return llm_response.choices[0].message.content

In [90]:
## Calling LLMprompt function to categorize query and parse query.

llm_query_parse_reply = query_parser_prompt_llm(user_query)
llm_query_parse_reply
# user_query

'{"query_type":"text","query_task_type":"question_answer","indicator":[],"aggregation_method":"","start_dt":"2025-01-01","end_dt":"2025-12-31","chart_required":"no"}'

In [91]:
import json

## To convert string to dictionary type
llm_query_parse_dict=json.loads(llm_query_parse_reply)
llm_query_parse_dict

{'query_type': 'text',
 'query_task_type': 'question_answer',
 'indicator': [],
 'aggregation_method': '',
 'start_dt': '2025-01-01',
 'end_dt': '2025-12-31',
 'chart_required': 'no'}

In [92]:
def fred_retrieval(llm_query_parse_dict):

  # Map the parser indicators to actual indicator labels
  indicator_map = {
        "gdp_rate": "GDP Growth Rate",
        "fed_fund_rate": "Federal Funds Rate",
        "employment_rate": "Employment Rate",
        "unemployment_rate": "Un-employment Rate",
        "inflation_5year": "Inflation Rate 5Year",
        "inflation_10year": "Inflation Rate 10Year"}

  start_dt = pd.to_datetime(llm_query_parse_dict['start_dt'])
  end_dt   = pd.to_datetime(llm_query_parse_dict['end_dt'])

  indicator_label=[indicator_map[i] for i in llm_query_parse_dict['indicator']]
  if llm_query_parse_dict['query_type']=='numeric':
    if llm_query_parse_dict['query_task_type']=='single':
      return(fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].mean())
    elif llm_query_parse_dict['query_task_type']=='timeseries':
      print('Timeseries query')
    else:
      print('Compare query')


qry_res=fred_retrieval(llm_query_parse_dict)
print(qry_res)

None


# <font color = yellow><b> Implementation of query intent identification and define function for each query type

### We have query_parser_prompt_llm(user_query) function that outputs query type and other parameters of query.
### Below are the query type identified by LLM :   1. text 2. numeric 3. hybrid
### Below are the intent of query identified by LLM :   1. single 2. timeseries 3. compare 4. summary 5. question_answer

####<font color = orange><b>Agent A : Query Intent Classification and Parser.

In [93]:
def query_intent_parser(user_query):
  ## Call Parser Prompt via LLM model to obtaain query type details.
  llm_query_parse_reply = query_parser_prompt_llm(user_query)
  ## To convert string to dictionary type
  import json
  llm_query_parse_dict=json.loads(llm_query_parse_reply)
  return(llm_query_parse_dict)


####<font color = orange><b>Agent B : Retrieval of data based on query type through orchestration of the query to different retreival functions.

##### Retrieval function 1 : Numeric and Single query type

In [94]:
## Designing of function for each query intent type.
## Below function is to be used for numeric type of query and with intent of single query.

def numeric_single_query_retreival(llm_query_parse_dict,user_query):

  # Map the parser indicators to actual indicator labels
  indicator_map = {
        "gdp_rate": "GDP Growth Rate",
        "fed_fund_rate": "Federal Funds Rate",
        "employment_rate": "Employment Rate",
        "unemployment_rate": "Un-employment Rate",
        "inflation_5year": "Inflation Rate 5Year",
        "inflation_10year": "Inflation Rate 10Year"}

  start_dt = pd.to_datetime(llm_query_parse_dict['start_dt'])
  end_dt   = pd.to_datetime(llm_query_parse_dict['end_dt'])

  indicator_label=[indicator_map[i] for i in llm_query_parse_dict['indicator']]
  numeric_single_query_answer_for_llm={}
  numeric_single_query_answer_for_llm['query_type']=llm_query_parse_dict['query_type']
  numeric_single_query_answer_for_llm['query_task_type']=llm_query_parse_dict['query_task_type']
  numeric_single_query_answer_for_llm['start_dt']=str(start_dt.date())
  numeric_single_query_answer_for_llm['end_dt']=str(end_dt.date())
  numeric_single_query_answer_for_llm['indicator']=indicator_label
  numeric_single_query_answer_for_llm['chart_required']='no'
  numeric_single_query_answer_for_llm['user_query']=user_query
  query_result=[]
  # To obtain mean of data.
  if llm_query_parse_dict['aggregation_method'].lower()=='mean':
    res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].mean()
  # To obtain a minimum of data
  elif llm_query_parse_dict['aggregation_method'].lower()=='min':
    res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].min()
   # To obtain a maximum of data
  elif llm_query_parse_dict['aggregation_method'].lower()=='max':
    res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].max()
   # To obtain latest data
  elif llm_query_parse_dict['aggregation_method'].lower()=='latest':
    res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].sort_values(by='Date').tail(1)
   # To obtain oldest data
  elif llm_query_parse_dict['aggregation_method'].lower()=='oldest':
    res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].sort_values(by='Date').head(1)
  # To obtain median of data
  elif llm_query_parse_dict['aggregation_method'].lower()=='median':
    res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].median()
  # To obtain standard deviation of data
  elif llm_query_parse_dict['aggregation_method'].lower()=='standard_deviation':
    res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].std()
  # To obtain percentile of data
  elif llm_query_parse_dict['aggregation_method'].lower()=='quantile':
    import re
    pattern='(\d+)(th|nd|st|rd)*\s*(percentile|quantile|percent|percentage|%)'
    match=re.search(pattern,user_query.lower())
    if match:
      if int(match.group(1))>=1 and int(match.group(1))<=100:
        value=int(match.group(1))
        val_decimal= round(value/100,2)
        res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].quantile(val_decimal)
        numeric_single_query_answer_for_llm['percentile_value']=value
      else:
          user_query_correct=input("""Please re-type query with correct percentile/quantile value ranging from 1-100, for example '20th percentile', '75 percentile', '90%' etc.
          """)
          match=re.search(pattern,user_query_correct.lower())
          value=int(match.group(1))
          val_decimal= round(value/100,2)
          res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].quantile(val_decimal)
          numeric_single_query_answer_for_llm['percentile_value']=value
          numeric_single_query_answer_for_llm['user_query']=user_query_correct
    else:
        user_query_correct=input("Please provide correct percentile/quantile value, for example '20th percentile', '75 percentile', '90%' etc.")
        match=re.search(pattern,user_query_correct.lower())
        value=int(match.group(1))
        val_decimal= round(value/100,2)
        res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].quantile(val_decimal)
        numeric_single_query_answer_for_llm['percentile_value']=value
        numeric_single_query_answer_for_llm['user_query']=user_query_correct
  # To get average if none provided
  else:
    res=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)].groupby('economic_indicator_desc')['rate_value_%'].mean()


  for i,row in res.reset_index().iterrows():
    query_result.append({'indicator_label':row['economic_indicator_desc'],'rate_value_%':round(row['rate_value_%'],2), 'unit':'percentage'})

  numeric_single_query_answer_for_llm['aggregation_method']=llm_query_parse_dict['aggregation_method']
  numeric_single_query_answer_for_llm['numeric_query_result']=query_result

  return(numeric_single_query_answer_for_llm)



##### Retrieval function 2 : Numeric and Timeseries/trend query type

In [95]:
## Designing of function for each query intent type.
## Below function is to be used for numeric type of query and with intent of trend/timeseries type query.

def numeric_timeseries_query_retreival(llm_query_parse_dict,user_query):

  # Map the parser indicators to actual indicator labels
  indicator_map = {
        "gdp_rate": "GDP Growth Rate",
        "fed_fund_rate": "Federal Funds Rate",
        "employment_rate": "Employment Rate",
        "unemployment_rate": "Un-employment Rate",
        "inflation_5year": "Inflation Rate 5Year",
        "inflation_10year": "Inflation Rate 10Year"}

  start_dt = pd.to_datetime(llm_query_parse_dict['start_dt'])
  end_dt   = pd.to_datetime(llm_query_parse_dict['end_dt'])

  indicator_label=[indicator_map[i] for i in llm_query_parse_dict['indicator']]

  # To get the list of timeseries data
  timeseries_data=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)][['rate_value_%','Date']]
  # To get the slop of trend/timeseries data
  x = np.arange(len(timeseries_data))
  slope = round(np.polyfit(x, timeseries_data['rate_value_%'], 1)[0],3)
  # to get total peaks(lows and ups) to identoify volatility.
  lows=0
  ups = 0
  for i in range(1,len(timeseries_data)-1):
    if (timeseries_data['rate_value_%'].iloc[i]>timeseries_data['rate_value_%'].iloc[i-1]) & \
      (timeseries_data['rate_value_%'].iloc[i]>timeseries_data['rate_value_%'].iloc[i+1]):
      ups = ups+1
    if (timeseries_data['rate_value_%'].iloc[i]<timeseries_data['rate_value_%'].iloc[i-1]) & \
      (timeseries_data['rate_value_%'].iloc[i]<timeseries_data['rate_value_%'].iloc[i+1]):
      lows = lows+1
  peaks=lows+ups
  peak_ratio = round(peaks/(len(timeseries_data)-2),3)
  # to get coefficient variation to identify volatility.
  cv = np.std(timeseries_data['rate_value_%'])/abs(np.mean(timeseries_data['rate_value_%']))
  # to prepare timeseries data output in dictionary format for llm
  timeseries_values_for_llm={}
  slop_rel=slope/abs(np.mean(timeseries_data['rate_value_%']))
  # To identofy the slope trend
  if  abs(slop_rel)<0.01:
    timeseries_values_for_llm['trend']='Stable'
  elif abs(slop_rel)<0.05:
    if slope>0:
      timeseries_values_for_llm['trend']='Moderate Upward'
    else:
      timeseries_values_for_llm['trend']='Moderate Downward'
  else:
    if slope>0:
      timeseries_values_for_llm['trend']='Strong Upward'
    else:
      timeseries_values_for_llm['trend']='Strong Downward'

  # To identify volatility
  if len(timeseries_data)<3:
    timeseries_values_for_llm['volatility']='Insufficient'
  else:
    if (cv<0.05) & (peak_ratio<0.10):
      timeseries_values_for_llm['volatility']='Low Volatility'
    elif (cv<0.10) & (peak_ratio<0.20):
      timeseries_values_for_llm['volatility']='Moderate Volatility'
    else:
      timeseries_values_for_llm['volatility']='High Volatility'

  # Chart visulaisation is required for facts.
  timeseries_values_for_llm['chart_required']='yes'
  # Add other required information for llm to generate fact data.
  timeseries_values_for_llm['user_query']=user_query
  timeseries_values_for_llm['query_task_type']=llm_query_parse_dict['query_task_type']
  timeseries_values_for_llm['high_value']=timeseries_data['rate_value_%'].max()
  timeseries_values_for_llm['low_value']=timeseries_data['rate_value_%'].min()
  timeseries_values_for_llm['average_value']=float(timeseries_data['rate_value_%'].mean())
  timeseries_values_for_llm['first_value']=float(timeseries_data['rate_value_%'].iloc[0])
  timeseries_values_for_llm['last_value']=float(timeseries_data['rate_value_%'].iloc[-1])
  timeseries_values_for_llm['slope_value']=float(slope)
  timeseries_values_for_llm['start_date']=str(start_dt.date())
  timeseries_values_for_llm['end_date']=str(end_dt.date())
  timeseries_values_for_llm['indicator']=indicator_label

  return timeseries_values_for_llm,timeseries_data



Retrieval function 3 : Numeric and multiple indicator Comparision query type

In [96]:
## Designing of function for each query intent type.
## Below function is to be used for numeric type of query and with intent of comparision between two indicators query.

def numeric_compare_multiindicator_query_retreival(llm_query_parse_dict):

  # Map the parser indicators to actual indicator labels
  indicator_map = {
        "gdp_rate": "GDP Growth Rate",
        "fed_fund_rate": "Federal Funds Rate",
        "employment_rate": "Employment Rate",
        "unemployment_rate": "Un-employment Rate",
        "inflation_5year": "Inflation Rate 5Year",
        "inflation_10year": "Inflation Rate 10Year"}

  start_dt = pd.to_datetime(llm_query_parse_dict['start_dt'])
  end_dt   = pd.to_datetime(llm_query_parse_dict['end_dt'])

  indicator_label=[indicator_map[i] for i in llm_query_parse_dict['indicator']]

  if len(indicator_label)>1:

  # To get the list of timeseries data of both indicator separately for comparision
    timeseries_data_1=fred_master_df[(fred_master_df['economic_indicator_desc']==(indicator_label[0])) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)][['rate_value_%','Date']]
    timeseries_data_2=fred_master_df[(fred_master_df['economic_indicator_desc']==(indicator_label[1])) & \
                (fred_master_df['Date']>=start_dt)\
                & (fred_master_df['Date']<=end_dt)][['rate_value_%','Date']]
  # To get the slop of both indicator separately for comparision
  x1 = np.arange(len(timeseries_data_1))
  slope1 = round(np.polyfit(x1, timeseries_data_1['rate_value_%'], 1)[0],3)
  x2= np.arange(len(timeseries_data_2))
  slope2 = round(np.polyfit(x2, timeseries_data_2['rate_value_%'], 1)[0],3)
  # to get total peaks(lows and ups) to identoify volatility.
  lows=0
  ups = 0
  for i in range(1,len(timeseries_data_1)-1):
    if (timeseries_data_1['rate_value_%'].iloc[i]>timeseries_data_1['rate_value_%'].iloc[i-1]) & \
      (timeseries_data_1['rate_value_%'].iloc[i]>timeseries_data_1['rate_value_%'].iloc[i+1]):
      ups = ups+1
    if (timeseries_data_1['rate_value_%'].iloc[i]<timeseries_data_1['rate_value_%'].iloc[i-1]) & \
      (timeseries_data_1['rate_value_%'].iloc[i]<timeseries_data_1['rate_value_%'].iloc[i+1]):
      lows = lows+1
  peaks1=lows+ups
  peak_ratio1 = round(peaks1/(len(timeseries_data_1)-2),3)
  lows=0
  ups = 0
  for i in range(1,len(timeseries_data_2)-1):
    if (timeseries_data_2['rate_value_%'].iloc[i]>timeseries_data_2['rate_value_%'].iloc[i-1]) & \
      (timeseries_data_2['rate_value_%'].iloc[i]>timeseries_data_2['rate_value_%'].iloc[i+1]):
      ups = ups+1
    if (timeseries_data_2['rate_value_%'].iloc[i]<timeseries_data_2['rate_value_%'].iloc[i-1]) & \
      (timeseries_data_2['rate_value_%'].iloc[i]<timeseries_data_2['rate_value_%'].iloc[i+1]):
      lows = lows+1
  peaks2=lows+ups
  peak_ratio2 = round(peaks2/(len(timeseries_data_2)-2),3)
  # to get coefficient variation to identify volatility.
  cv1 = np.std(timeseries_data_1['rate_value_%'])/abs(np.mean(timeseries_data_1['rate_value_%']))
  cv2 = np.std(timeseries_data_2['rate_value_%'])/abs(np.mean(timeseries_data_2['rate_value_%']))
  # to prepare timeseries data output in dictionary format for llm
  comparision_values_for_llm={}

  # Add other required information for llm to generate fact data.

  comparision_values_for_llm['type_of_comparision']='multiple_indicator'
  comparision_values_for_llm['indicator_1']={'indicator_label':indicator_label[0],\
                                             'high_value':timeseries_data_1['rate_value_%'].max(),\
                                             'low_value':timeseries_data_1['rate_value_%'].min(),\
                                             'average_value':round(float(timeseries_data_1['rate_value_%'].mean()),2),\
                                             'first_value':float(timeseries_data_1['rate_value_%'].iloc[0]),\
                                             'last_value':float(timeseries_data_1['rate_value_%'].iloc[-1]),\
                                             'slope_value':float(slope1),\
                                             'start_date':str(start_dt.date()),\
                                             'end_date':str(end_dt.date())
                                            }
  comparision_values_for_llm['indicator_2']={'indicator_label':indicator_label[1],\
                                             'high_value':timeseries_data_2['rate_value_%'].max(),\
                                             'low_value':timeseries_data_2['rate_value_%'].min(),\
                                             'average_value':round(float(timeseries_data_2['rate_value_%'].mean()),2),\
                                             'first_value':float(timeseries_data_2['rate_value_%'].iloc[0]),\
                                             'last_value':float(timeseries_data_2['rate_value_%'].iloc[-1]),\
                                             'slope_value':float(slope2),\
                                             'start_date':str(start_dt.date()),\
                                             'end_date':str(end_dt.date())
                                            }
  # Chart visulaisation is required for facts.
  comparision_values_for_llm['chart_required']='yes'

  slop_rel1=slope1/abs(np.mean(timeseries_data_1['rate_value_%']))
  slop_rel2=slope1/abs(np.mean(timeseries_data_2['rate_value_%']))

  # To identify the slope trend for first indicator
  if  abs(slop_rel1)<0.01:
    comparision_values_for_llm['indicator_1']['trend']='Stable'
  elif abs(slop_rel1)<0.05:
    if slope1>0:
      comparision_values_for_llm['indicator_1']['trend']='Moderate Upward'
    else:
      comparision_values_for_llm['indicator_1']['trend']='Moderate Downward'
  else:
    if slope1>0:
      comparision_values_for_llm['indicator_1']['trend']='Strong Upward'
    else:
      comparision_values_for_llm['indicator_1']['trend']='Strong Downward'

    # To identify the slope trend for second indicator
  if  abs(slop_rel2)<0.01:
    comparision_values_for_llm['indicator_2']['trend']='Stable'
  elif abs(slop_rel2)<0.05:
    if slope2>0:
      comparision_values_for_llm['indicator_2']['trend']='Moderate Upward'
    else:
      comparision_values_for_llm['indicator_2']['trend']='Moderate Downward'
  else:
    if slope2>0:
      comparision_values_for_llm['indicator_2']['trend']='Strong Upward'
    else:
      comparision_values_for_llm['indicator_2']['trend']='Strong Downward'

  # To identify volatility for first indicator
  if len(timeseries_data_1)<3:
    comparision_values_for_llm['indicator_1']['volatility']='Insufficient'
  else:
    if (cv1<0.05) & (peak_ratio1<0.10):
      comparision_values_for_llm['indicator_1']['volatility']='Low Volatility'
    elif (cv1<0.10) & (peak_ratio1<0.20):
      comparision_values_for_llm['indicator_1']['volatility']='Moderate Volatility'
    else:
      comparision_values_for_llm['indicator_1']['volatility']='High Volatility'


  # To identify volatility for second indicator
  if len(timeseries_data_2)<3:
    comparision_values_for_llm['indicator_2']['volatility']='Insufficient'
  else:
    if (cv2<0.05) & (peak_ratio2<0.10):
      comparision_values_for_llm['indicator_2']['volatility']='Low Volatility'
    elif (cv2<0.10) & (peak_ratio2<0.20):
      comparision_values_for_llm['indicator_2']['volatility']='Moderate Volatility'
    else:
      comparision_values_for_llm['indicator_2']['volatility']='High Volatility'

  # To identify higher of average values of two indicators for comparision metrix
  if timeseries_data_1['rate_value_%'].mean()>timeseries_data_2['rate_value_%'].mean():
    comparision_values_for_llm['indicator_higher_rate']=indicator_label[0]
    comparision_values_for_llm['difference_of_indicators_average']=round(float((timeseries_data_1['rate_value_%'].mean())-(timeseries_data_2['rate_value_%'].mean())),2)
  else:
    comparision_values_for_llm['indicator_higher_rate']=indicator_label[1]
    comparision_values_for_llm['difference_of_indicators_average']=round(float((timeseries_data_2['rate_value_%'].mean())-(timeseries_data_1['rate_value_%'].mean())),2)

  return comparision_values_for_llm,timeseries_data_1,timeseries_data_2



Retrieval function 4 : Numeric and multi duration Comparision query type

In [97]:
## Designing of function for each query intent type.
## Below function is to be used for numeric type of query and with intent of comparision between two period range query.

def numeric_compare_multiperiod_query_retreival(llm_query_parse_dict):

  # Map the parser indicators to actual indicator labels
  indicator_map = {
        "gdp_rate": "GDP Growth Rate",
        "fed_fund_rate": "Federal Funds Rate",
        "employment_rate": "Employment Rate",
        "unemployment_rate": "Un-employment Rate",
        "inflation_5year": "Inflation Rate 5Year",
        "inflation_10year": "Inflation Rate 10Year"}

  start_dt = pd.to_datetime(llm_query_parse_dict['start_dt'])
  end_dt   = pd.to_datetime(llm_query_parse_dict['end_dt'])

  indicator_label=[indicator_map[i] for i in llm_query_parse_dict['indicator']]

    # To get the list of timeseries data of both time period separately for comparision
  timeseries_data_1=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt[0])\
                & (fred_master_df['Date']<=end_dt[0])][['rate_value_%','Date']]
  timeseries_data_2=fred_master_df[(fred_master_df['economic_indicator_desc'].isin(indicator_label)) & \
                (fred_master_df['Date']>=start_dt[1])\
                & (fred_master_df['Date']<=end_dt[1])][['rate_value_%','Date']]
  # To get the slop of both duration data separately for comparision
  x1 = np.arange(len(timeseries_data_1))
  slope1 = round(np.polyfit(x1, timeseries_data_1['rate_value_%'], 1)[0],3)
  x2= np.arange(len(timeseries_data_2))
  slope2 = round(np.polyfit(x2, timeseries_data_2['rate_value_%'], 1)[0],3)
  # to get total peaks(lows and ups) to identoify volatility.
  lows=0
  ups = 0
  for i in range(1,len(timeseries_data_1)-1):
    if (timeseries_data_1['rate_value_%'].iloc[i]>timeseries_data_1['rate_value_%'].iloc[i-1]) & \
      (timeseries_data_1['rate_value_%'].iloc[i]>timeseries_data_1['rate_value_%'].iloc[i+1]):
      ups = ups+1
    if (timeseries_data_1['rate_value_%'].iloc[i]<timeseries_data_1['rate_value_%'].iloc[i-1]) & \
      (timeseries_data_1['rate_value_%'].iloc[i]<timeseries_data_1['rate_value_%'].iloc[i+1]):
      lows = lows+1
  peaks1=lows+ups
  peak_ratio1 = round(peaks1/(len(timeseries_data_1)-2),3)
  lows=0
  ups = 0
  for i in range(1,len(timeseries_data_2)-1):
    if (timeseries_data_2['rate_value_%'].iloc[i]>timeseries_data_2['rate_value_%'].iloc[i-1]) & \
      (timeseries_data_2['rate_value_%'].iloc[i]>timeseries_data_2['rate_value_%'].iloc[i+1]):
      ups = ups+1
    if (timeseries_data_2['rate_value_%'].iloc[i]<timeseries_data_2['rate_value_%'].iloc[i-1]) & \
      (timeseries_data_2['rate_value_%'].iloc[i]<timeseries_data_2['rate_value_%'].iloc[i+1]):
      lows = lows+1
  peaks2=lows+ups
  peak_ratio2 = round(peaks2/(len(timeseries_data_2)-2),3)
  # to get coefficient variation to identify volatility.
  cv1 = np.std(timeseries_data_1['rate_value_%'])/abs(np.mean(timeseries_data_1['rate_value_%']))
  cv2 = np.std(timeseries_data_2['rate_value_%'])/abs(np.mean(timeseries_data_2['rate_value_%']))
  # to prepare timeseries data output in dictionary format for llm
  comparision_values_for_llm={}

  # Add other required information for llm to generate fact data.

  comparision_values_for_llm['type_of_comparision']='compare_multiple_time_period'
  comparision_values_for_llm['period_1']={'indicator_label':indicator_label,\
                                             'high_value':timeseries_data_1['rate_value_%'].max(),\
                                             'low_value':timeseries_data_1['rate_value_%'].min(),\
                                             'average_value':round(float(timeseries_data_1['rate_value_%'].mean()),2),\
                                             'first_value':float(timeseries_data_1['rate_value_%'].iloc[0]),\
                                             'last_value':float(timeseries_data_1['rate_value_%'].iloc[-1]),\
                                             'slope_value':float(slope1),\
                                             'period1_range':str(start_dt[0].date())+' to '+str(end_dt[0].date())
                                            }
  comparision_values_for_llm['period_2']={'indicator_label':indicator_label,\
                                             'high_value':timeseries_data_2['rate_value_%'].max(),\
                                             'low_value':timeseries_data_2['rate_value_%'].min(),\
                                             'average_value':round(float(timeseries_data_2['rate_value_%'].mean()),2),\
                                             'first_value':float(timeseries_data_2['rate_value_%'].iloc[0]),\
                                             'last_value':float(timeseries_data_2['rate_value_%'].iloc[-1]),\
                                             'slope_value':float(slope2),\
                                             'period2_range':str(start_dt[1].date())+' to '+str(end_dt[1].date())
                                            }
  # Chart visulaisation is required for facts.
  comparision_values_for_llm['chart_required']='yes'

  slop_rel1=slope1/abs(np.mean(timeseries_data_1['rate_value_%']))
  slop_rel2=slope1/abs(np.mean(timeseries_data_2['rate_value_%']))

  # To identify the slope trend for first indicator
  if  abs(slop_rel1)<0.01:
    comparision_values_for_llm['period_1']['trend']='Stable'
  elif abs(slop_rel1)<0.05:
    if slope1>0:
      comparision_values_for_llm['period_1']['trend']='Moderate Upward'
    else:
      comparision_values_for_llm['period_1']['trend']='Moderate Downward'
  else:
    if slope1>0:
      comparision_values_for_llm['period_1']['trend']='Strong Upward'
    else:
      comparision_values_for_llm['period_1']['trend']='Strong Downward'

    # To identify the slope trend for second indicator
  if  abs(slop_rel2)<0.01:
    comparision_values_for_llm['period_2']['trend']='Stable'
  elif abs(slop_rel2)<0.05:
    if slope2>0:
      comparision_values_for_llm['period_2']['trend']='Moderate Upward'
    else:
      comparision_values_for_llm['period_2']['trend']='Moderate Downward'
  else:
    if slope2>0:
      comparision_values_for_llm['period_2']['trend']='Strong Upward'
    else:
      comparision_values_for_llm['period_2']['trend']='Strong Downward'

  # To identify volatility for first indicator
  if len(timeseries_data_1)<3:
    comparision_values_for_llm['period_1']['volatility']='Insufficient'
  else:
    if (cv1<0.05) & (peak_ratio1<0.10):
      comparision_values_for_llm['period_1']['volatility']='Low Volatility'
    elif (cv1<0.10) & (peak_ratio1<0.20):
      comparision_values_for_llm['period_1']['volatility']='Moderate Volatility'
    else:
      comparision_values_for_llm['period_1']['volatility']='High Volatility'


  # To identify volatility for second indicator
  if len(timeseries_data_2)<3:
    comparision_values_for_llm['period_2']['volatility']='Insufficient'
  else:
    if (cv2<0.05) & (peak_ratio2<0.10):
      comparision_values_for_llm['period_2']['volatility']='Low Volatility'
    elif (cv2<0.10) & (peak_ratio2<0.20):
      comparision_values_for_llm['period_2']['volatility']='Moderate Volatility'
    else:
      comparision_values_for_llm['period_2']['volatility']='High Volatility'

  # To identify higher of average values of two indicators for comparision metrix
  if timeseries_data_1['rate_value_%'].mean()>timeseries_data_2['rate_value_%'].mean():
    comparision_values_for_llm['period_having_higher_rate']=str(start_dt[0].date())+' to '+str(end_dt[0].date())
    comparision_values_for_llm['difference_of_rate_average']=round(float((timeseries_data_1['rate_value_%'].mean())-(timeseries_data_2['rate_value_%'].mean())),2)
  else:
    comparision_values_for_llm['period_having_higher_rate']=str(start_dt[1].date())+' to '+str(end_dt[1].date())
    comparision_values_for_llm['difference_of_rate_average']=round(float((timeseries_data_2['rate_value_%'].mean())-(timeseries_data_1['rate_value_%'].mean())),2)

  # To add inidicator values in both period range for llm
  comparision_values_for_llm['period_1']['indicator_label']=indicator_label
  comparision_values_for_llm['period_2']['indicator_label']=indicator_label

  return comparision_values_for_llm,timeseries_data_1,timeseries_data_2



#### Retrieval function 5 : Text and document summary query type

In [98]:
## Designing of function for each query intent type.
## Below function is to be used for text type of query and with intent of document summarisation query.

def text_document_summary_query_retreival(llm_query_parse_dict):

  # Map the parser indicators to actual indicator labels
  indicator_map = {
        "gdp_rate": "GDP Growth Rate",
        "fed_fund_rate": "Federal Funds Rate",
        "employment_rate": "Employment Rate",
        "unemployment_rate": "Un-employment Rate",
        "inflation_5year": "Inflation Rate 5Year",
        "inflation_10year": "Inflation Rate 10Year"}

  start_dt = pd.to_datetime(llm_query_parse_dict['start_dt'])
  end_dt   = pd.to_datetime(llm_query_parse_dict['end_dt'])
  text_summary_doc_for_llm={}
  text_summary_doc_for_llm['chart_required']='no'
  indicator_label=[indicator_map[i] for i in llm_query_parse_dict['indicator']]
  if llm_query_parse_dict['query_task_type']=='summary_fomc_minute':
    summary_doc_df = fomc_chunk_new_df[(fomc_chunk_new_df.doc_type=='Minute') & (fomc_chunk_new_df.meeting_date>=start_dt) & \
                                                                              (fomc_chunk_new_df.meeting_date<=end_dt)]
    retreievd_doc_with_timestamp = pd.DataFrame(summary_doc_df.groupby('meeting_date')['chunked_text'].apply(lambda x: "\n\n".join(x)))
    retreievd_doc_with_timestamp.reset_index(inplace=True)
    retreievd_doc_with_timestamp['meeting_date']=pd.to_datetime(retreievd_doc_with_timestamp['meeting_date']).dt.strftime("%Y-%m-%d")
    retreievd_doc_with_timestamp=retreievd_doc_with_timestamp.to_dict(orient="records")
    text_summary_doc_for_llm['document_retreived_for_sumamry']=retreievd_doc_with_timestamp
    text_summary_doc_for_llm['query_type']='document_summary'
    text_summary_doc_for_llm['document_type']='FOMC_minute'
    text_summary_doc_for_llm['start_date']=str(start_dt.date())
    text_summary_doc_for_llm['end_date']=str(end_dt.date())
    return text_summary_doc_for_llm
  elif llm_query_parse_dict['query_task_type']=='summary_fomc_statement':
    summary_doc_df = fomc_chunk_new_df[(fomc_chunk_new_df.doc_type=='Statement') & (fomc_chunk_new_df.meeting_date>=start_dt) & \
                                                                                  (fomc_chunk_new_df.meeting_date<=end_dt)]
    retreievd_doc_with_timestamp = pd.DataFrame(summary_doc_df.groupby('meeting_date')['chunked_text'].apply(lambda x: "\n\n".join(x)))
    retreievd_doc_with_timestamp.reset_index(inplace=True)
    retreievd_doc_with_timestamp['meeting_date']=pd.to_datetime(retreievd_doc_with_timestamp['meeting_date']).dt.strftime("%Y-%m-%d")
    retreievd_doc_with_timestamp=retreievd_doc_with_timestamp.to_dict(orient="records")
    text_summary_doc_for_llm['document_retreived_for_sumamry']=retreievd_doc_with_timestamp
    text_summary_doc_for_llm['query_type']='document_summary'
    text_summary_doc_for_llm['document_type']='FOMC_statement'
    text_summary_doc_for_llm['start_date']=str(start_dt.date())
    text_summary_doc_for_llm['end_date']=str(end_dt.date())
    return text_summary_doc_for_llm





#### Retrieval function 6 : Text and topic wise summary query type

In [99]:
## Designing of function for each query intent type.
## Below function is to be used for text type of query and with intent of topic wise summarisation query.

def text_topic_summary_query_retreival(llm_query_parse_dict,user_query):

  # Map the parser indicators to actual indicator labels
  indicator_map = {
        "gdp_rate": "GDP Growth Rate",
        "fed_fund_rate": "Federal Funds Rate",
        "employment_rate": "Employment Rate",
        "unemployment_rate": "Un-employment Rate",
        "inflation_5year": "Inflation Rate 5Year",
        "inflation_10year": "Inflation Rate 10Year"}

  start_dt = pd.to_datetime(llm_query_parse_dict['start_dt'])
  end_dt   = pd.to_datetime(llm_query_parse_dict['end_dt'])
  indicator_label=[indicator_map[i] for i in llm_query_parse_dict['indicator']]
  text_summary_item_for_llm={}
  text_summary_item_for_llm['chart_required']='no'
  ## Call query rewriter function to generate efficient query.
  llm_output=query_rewriter_llm(user_query)
  ## Call collection load function for semantic retrieval of text.
  chroma_collection=fomc_collection_load()
  ## Call query engine function to retrieve top 15 semantic chunks from embed collection.
  top_15_similar_document=query_engine(llm_output['rewritten_query'],chroma_collection)
  ## Calling re-rank function to get normalised re-ranked score of top 15 documents for refined search result.
  top_15_doc_reranked_score_df= rerank_top15_document(llm_output['rewritten_query'],top_15_similar_document)
  ## Return top 10 documents based on Encoder score for llm to provide summary of chunks based on user topic.
  document_retreived_for_sumamry = top_15_doc_reranked_score_df.sort_values(by='Encoder_Norm_score',ascending=False)[0:10]
  text_summary_item_for_llm['query_type']='topic_summary'
  text_summary_item_for_llm['original_user_query']=user_query
  text_summary_item_for_llm['rewritten_user_query']=llm_output['rewritten_query']

  metadatas_list = document_retreived_for_sumamry['Metadatas'].tolist()
  documents_list = document_retreived_for_sumamry['Document'].tolist()

  combined_retrieval_list = []

  for meta, doc in zip(metadatas_list, documents_list):
    combined_retrieval_list.append({**meta,"document_text": doc})

  text_summary_item_for_llm['documents_retrieved_for_summary'] = combined_retrieval_list
  return text_summary_item_for_llm




#### Retrieval function 7 : Text and question answer query type

In [100]:
## Designing of function for each query intent type.
## Below function is to be used for text type of query and with intent of qyestion answer type query.

def text_question_answer_query_retreival(llm_query_parse_dict,user_query):

  # Map the parser indicators to actual indicator labels
  indicator_map = {
        "gdp_rate": "GDP Growth Rate",
        "fed_fund_rate": "Federal Funds Rate",
        "employment_rate": "Employment Rate",
        "unemployment_rate": "Un-employment Rate",
        "inflation_5year": "Inflation Rate 5Year",
        "inflation_10year": "Inflation Rate 10Year"}

  start_dt = pd.to_datetime(llm_query_parse_dict['start_dt'])
  end_dt   = pd.to_datetime(llm_query_parse_dict['end_dt'])
  indicator_label=[indicator_map[i] for i in llm_query_parse_dict['indicator']]
  text_question_answer_for_llm={}
  text_question_answer_for_llm['chart_required']='no'
  ## Call query rewriter function to generate efficient query.
  llm_output=query_rewriter_llm(user_query)
  ## Call collection load function for semantic retrieval of text.
  chroma_collection=fomc_collection_load()
  ## Call query engine function to retrieve top 15 semantic chunks from embed collection.
  top_15_similar_document=query_engine(llm_output['rewritten_query'],chroma_collection)
  ## Calling re-rank function to get normalised re-ranked score of top 15 documents for refined search result.
  top_15_doc_reranked_score_df= rerank_top15_document(llm_output['rewritten_query'],top_15_similar_document)
  ## Return top 10 documents based on Encoder score for llm to provide summary of chunks based on user topic.
  document_retreived_for_query = top_15_doc_reranked_score_df.sort_values(by='Encoder_Norm_score',ascending=False)[0:4]
  text_question_answer_for_llm['query_type']=llm_query_parse_dict['query_task_type']
  text_question_answer_for_llm['original_user_query']=user_query
  text_question_answer_for_llm['rewritten_user_query']=llm_output['rewritten_query']

  metadatas_list = document_retreived_for_query['Metadatas'].tolist()
  documents_list = document_retreived_for_query['Document'].tolist()

  combined_retrieval_list = []

  for meta, doc in zip(metadatas_list, documents_list):
    combined_retrieval_list.append({**meta,"document_text": doc})

  text_question_answer_for_llm['documents_retrieved_for_question_answer'] = combined_retrieval_list
  return text_question_answer_for_llm

#### Function to call chroma database collection for semantic retrieval.

In [101]:
## Import chromadb for stroing of embedded chunks
!pip install -U -q chromadb
import chromadb, shutil, os, io, contextlib

from chromadb.utils import embedding_functions

def fomc_collection_load():
  # Mount google drive
  from google.colab import drive
  with contextlib.redirect_stdout(io.StringIO()):
    drive.mount('/content/drive')

  # # Remove any existing chroma db in local path if any.
  # if os.path.exists('/content/FOMC_ChromaDB_Data_v2'):
  #   shutil.rmtree('/content/FOMC_ChromaDB_Data_v2')

  # # Copy complete folder to local Colab path
  # shutil.copytree(
  #   "/content/drive/MyDrive/FOMC_ChromaDB_Data_v2",
  #   "/content/FOMC_ChromaDB_Data_v2"
  #   )

  from chromadb.utils import embedding_functions
  embed_model = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="mukaj/fin-mpnet-base")
  # Local path drive details for chromadb upload.
  chroma_drive_path = "/content/FOMC_ChromaDB_Data_v2"

  # instantiating PersistentClient of chromadb.
  client = chromadb.PersistentClient(path=chroma_drive_path)

  # Defining collection set up in chroma to pass the embedding function for embedding document chunks.
  chroma_collection = client.get_or_create_collection(name='FOMC_Chroma_Client1', embedding_function=embed_model)

  return chroma_collection


### Function to rewrite user query for effective semantic retrieval through llm.

In [102]:
## Function to rewrite user query using claude sonnet llm for generating efficient query.

!pip install anthropic
import anthropic, io, contextlib

def query_rewriter_llm(user_query):
  # Mount Google Drive.
  from google.colab import drive
  with contextlib.redirect_stdout(io.StringIO()):
    drive.mount('/content/drive')

  # Set the anthropic claude API key
  filepath = "/content/drive/MyDrive/Agentic RAG Project/"
  with open(filepath + "claude_anthropic_api_key.txt", "r") as f:
   api_key = ' '.join(f.readlines())
   client = anthropic.Anthropic(api_key=api_key)

  original_query=user_query
  response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=150,
    system="""You are an expert in Federal Reserve communications and monetary policy domain and excel at rewriting original query to generate optimized output query.\
    You should rewrite the user's original query for effective searching in FOMC meeting minutes and policy statements.\
    Your primary aim is to expand relevant financial terminology, provide synonyms, and make the intent explicit.\
     You must return only the rewritten query in the output and do not add any labels, do not provide an explanation, and do not add the original query.""",
    messages=[
        {"role": "user", "content": f"Rewrite this query for better retrieval: {original_query}"}
    ])
  llm_output={"rewritten_query":response.content[0].text, "original_query":original_query}
  return llm_output

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 20.5 MB/s eta 0:00:00


#### Function for query search in chroma db collection and retrieval of top 15 semantic chunks.

In [103]:
## Defining Function for query search in chroma vector DB.

def query_engine(user_query,chroma_collection):

  ## Perform query on choromadb based on user query and retrieve top 15 relevant search from vector db.
    query_results = chroma_collection.query(query_texts=user_query,n_results=15)
  ## Creating Dataframe of query results.
    query_results_df=pd.DataFrame(columns=['Document','Metadatas','Distance'])
    query_results_df['Document']=query_results['documents'][0]
    query_results_df['Metadatas']=query_results['metadatas'][0]
    query_results_df['Distance']=query_results['distances'][0]
    query_results_df['Document_ID']=[i['doc_id'] for i in query_results_df['Metadatas']]
    query_results_df['Chunk_ID']=[i['chunk_id'] for i in query_results_df['Metadatas']]
    query_results_df['Meeting_Date']=[i['meeting_date'] for i in query_results_df['Metadatas']]
  ## Return sorted table on Distance.
    top_15_similar_document= query_results_df.sort_values(by='Distance',ascending=True)
    return(top_15_similar_document)


#### Re-ranking function to rerank top 15 chunks retrieved at first instance based on similarity scores for further refinement.

In [104]:
## Re-ranking function using crossencoder model for retrieval of more semantic similar documents from top 15 docuemnts retreievd from chromadb.

from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/stsb-roberta-large")
def rerank_top15_document(user_query,top_15_similar_document):

  ## Creating pair of document(top 10 documents obtianed from engiene query) and query list for re-ranking score.
    cross_encoder_sentence_query_pair = [[user_query,x] for x in top_15_similar_document.Document]

  ## Predicting scores of document and query pair.
    score=reranker.predict(cross_encoder_sentence_query_pair)

  ## Normalising score obtained above in range[0-1] by applying sigmoid function.
    score_norm = [round(float(1/(1+np.exp(-scr))),4) for scr in score]

  ## Updating score and normalised score in dataframe.
    top_15_similar_document['Encoder_Norm_score']=score_norm
    top_15_similar_document['Encoder_score']=score

  ## return sorted table by sorting Norm Score in Descending order.
    return(top_15_similar_document.sort_values(by='Encoder_Norm_score',ascending=False))


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### Primary Orchestrator function for retreival from required tool.

In [105]:
### Orchestartion of query to call required functions for retrieval.

def query_orchestrator_for_retreival(llm_query_parse_dict,user_query):
  if llm_query_parse_dict['query_type']=='numeric':   ## Route numeric query type to correct functions for retrieval.
    if llm_query_parse_dict['query_task_type']=='single':
      qry_res=numeric_single_query_retreival(llm_query_parse_dict,user_query)
      return qry_res
    elif llm_query_parse_dict['query_task_type']=='timeseries':
      timeseries_values_for_llm,timeseries_data=numeric_timeseries_query_retreival(llm_query_parse_dict,user_query)
      return timeseries_values_for_llm,timeseries_data
    elif llm_query_parse_dict['query_task_type']=='compare_indicator':
      comparision_values_for_llm,timeseries_data_1,timeseries_data_2=numeric_compare_multiindicator_query_retreival(llm_query_parse_dict)
      return comparision_values_for_llm,timeseries_data_1,timeseries_data_2
    elif llm_query_parse_dict['query_task_type']=='compare_period':
      comparision_values_for_llm,timeseries_data_1,timeseries_data_2=numeric_compare_multiperiod_query_retreival(llm_query_parse_dict)
      return comparision_values_for_llm,timeseries_data_1,timeseries_data_2
  ## Route text query type to correct functions for retrieval
  elif llm_query_parse_dict['query_type']=='text':
      if (llm_query_parse_dict['query_task_type']=='summary_fomc_minute') or (llm_query_parse_dict['query_task_type']=='summary_fomc_statement'):
        if not(llm_query_parse_dict['start_dt']) and not(llm_query_parse_dict['end_dt']):
          time_range=input('Please provide year of document for summary:')
          llm_query_parse_dict=query_intent_parser(user_query+time_range)
          documents_for_llm_summary=text_document_summary_query_retreival(llm_query_parse_dict)
          return documents_for_llm_summary
        else:
          documents_for_llm_summary=text_document_summary_query_retreival(llm_query_parse_dict)
          return documents_for_llm_summary
      elif llm_query_parse_dict['query_task_type']=='summary_topic':
        documents_for_llm_summary=text_topic_summary_query_retreival(llm_query_parse_dict,user_query)
        return documents_for_llm_summary
      elif llm_query_parse_dict['query_task_type']=='question_answer':
        documents_for_llm_summary=text_question_answer_query_retreival(llm_query_parse_dict,user_query)
        return documents_for_llm_summary


In [106]:
## To check all the query type functions are orchestrator function is working as expected.

user_query=input('input your query : ')
query_parse=query_intent_parser(user_query)
query_orchestrator_for_retreival(query_parse,user_query)
# query_parse



input your query : what was gdp rate in 2024


{'query_type': 'numeric',
 'query_task_type': 'single',
 'start_dt': '2024-01-01',
 'end_dt': '2024-12-31',
 'indicator': ['GDP Growth Rate'],
 'chart_required': 'no',
 'user_query': 'what was gdp rate in 2024',
 'aggregation_method': 'Average',
 'numeric_query_result': [{'indicator_label': 'GDP Growth Rate',
   'rate_value_%': 3.01,
   'unit': 'percentage'}]}

# <font color = yellow><b> Implementation of Response generation Agent using LLM prompt

In [107]:
## To mount google drive in colab for api keys load.
import contextlib

import io
from google.colab import drive

with contextlib.redirect_stdout(io.StringIO()):
    drive.mount('/content/drive')
# Set the API key
filepath = "/content/drive/MyDrive/Agentic RAG Project/"
with open(filepath + "OpenAI_API_Key.txt", "r") as f:
 api_key = ' '.join(f.readlines())


#### LLM Prompt 1: For generation of response to numeric and single type of query.

In [108]:
## Creating system message and user message for prompt.

def numeric_single_query_response_llm(output_json,api_key):
  system_message = """ You are an expert financial analysis assistant who is expertise in Macroeconomic analysis domain especially with FRED economic indcator and FOMC to generate human understandable response from user query. """

  user_message = f"""
  You are an expert financial analysis assistant for FOMC and FRED-based macroeconomic queries.
  You will be provided with structured query details and the computed numeric result in JSON format.
  Your task is to generate a clear, human understandable, natural language response to the user's question using only from the provided inputs.

  You must follow below guidelines before generating the response-
  1. Read and understand the user question from <<user_query>> parameter.
  2. Understand the user question context from below parameter values:
	  a. <<query_task_type>>
	  b. <<start_date>>
	  c. <<end_date>>
	  d. <<indicator>>
	  e. <<aggregation_method>>
  3. Read the final computed answer from <<numeric_query_result>>.
  4. Generate a concise and natural response that directly answers the user question.
  5. Do not calculate or generate any value on your own.
  6. Use only the result values present in <<numeric_query_result>>.
  7. For better response, you must mention the indicator name and unit in the response.
  8. When required mention the <<aggregation_method>> naturally in the response.
  9. If the numeric result is empty, say that no data was found for the requested query period.
  10. Replace the word percentage with '%' when generating direct response to user query after a single space.

 Note that query and response details is provided in {output_json} and you must return a response.
  """
  # Initialize client with API key
  client = OpenAI(api_key=api_key)

  ## Call LLM model GPT-5.4 to generate query parser in JSON response. We arere using chat gpt model GPT-4o Realtime(gpt-4.5-preview).

  llm_response_generation = openai.chat.completions.create( model="gpt-5.4",
                  messages=[{"role":"system", "content":system_message},
                             {"role":"user","content":user_message}], temperature=0
                  )
  return llm_response_generation.choices[0].message.content


#### LLM Prompt 2: For generation of response to numeric and timeseries type of query.

In [109]:
## Creating system message and user message for prompt.

def numeric_timeseries_query_response_llm(output_json,api_key):
  system_message = """ You are an expert financial analysis assistant who is expertise in Macroeconomic analysis domain especially with FRED economic indcator and FOMC to generate human understandable response from user query. """

  user_message = f"""
  You are an expert financial analysis assistant for FOMC and FRED-based macroeconomic queries.
  You will be provided with structured query details and the computed numeric result in JSON format.
  Your task is to generate a clear, human understandable, natural language response to the user's question using only from the provided inputs.

  You must follow below guidelines before generating the response-
  1. Read and understand the user question from <<user_query>> parameter.
  2. Understand the user question context from below parameter values:
	  a. <<query_task_type>>
	  b. <<start_date>>
	  c. <<end_date>>
	  d. <<indicator>>
	  e. <<high_value>>
	  f. <<low_value>>
    g. <<average_value>>
    h. <<first_value>>
	  i. <<last_value>>
	  j. <<slope_value>>
	  k. <<trend>>
	  l. <<volatility>>
	  m. <<chart_required>>

  3. Generate a concise and natural response that answers the user question directly and naturally.
  4. Do not calculate or generate any value on your own.
  5. You must generate natural language answers that explains the trend of the data in simple language.
  6. When required mention the <<trend>> and <<volatility>> naturally in the response.
  7. If the result is empty or missing, say that no data was found for the requested query period.
  8. Return plain text only and do not format in bold font. Do not use Markdown formatting like **bold** or bullet points.

  Note that query and response details is provided in {output_json} and you must return a response.
  """
  # Initialize client with API key
  client = OpenAI(api_key=api_key)

  ## Call LLM model GPT-5.4 to generate query parser in JSON response. We arere using chat gpt model GPT-4o Realtime(gpt-4.5-preview).

  llm_response_generation = openai.chat.completions.create( model="gpt-5.4",
                  messages=[{"role":"system", "content":system_message},
                             {"role":"user","content":user_message}], temperature=0
                  )
  return llm_response_generation.choices[0].message.content


#### LLM Prompt 3: For generation of response to numeric and indicator comparision type of query.

In [110]:
## Creating system message and user message for prompt.

def numeric_indicator_compare_query_response_llm(output_json,api_key):
  system_message = """ You are an expert financial analysis assistant who is expertise in Macroeconomic analysis domain especially with FRED economic indcator and FOMC to generate human understandable response from user query. """

  user_message = f"""
  You are an expert financial analysis assistant for FOMC and FRED-based macroeconomic queries.
  You will be provided with structured query details and the computed numeric result in JSON format.
  Your task is to perform comparison analysis between two indicators and generate a clear, human understandable, natural language comparative analysis response to the user's question using only from the provided structured input data.

  You must follow below guidelines before generating the response-
  1. Read and understand the user question from <<user_query>> parameter.
  2. Understand the user question context from <<type_of_comparision>> parameter.
  3. Read and perform comparison analysis between indicators from below parameter values provided for both the indicators:
    a. <<indicator_label>>
    b. <<high_value>>
    c. <<low_value>>
    d. <<average_value>>
    e. <<first_value>>
    f. <<last_value>>
    g. <<slope_value>>
    h. <<start_date>>
    i. <<end_date>>
    j. <<trend>>
    k. <<volatility>>
  4. Read the value of parameter <<'indicator_higher_rate'>> to understand which indicator have higher rate value from the two use this information when generating a comparison summary.
  5. Parameter <<difference_of_indicators_average>> will tell you the value of rate differences between two indicators and use this information when generating a comparison summary.
  6. You must compare the indicators based on their trend, volatility, average value and slope.
  7. If indicator have significantly increased or decreased, then mention the same by highlighting during the response generation.
  8. A chart will be required to visualise the trend and pattern of both indicators in given time range and mention that a visual comparison chart is provided.
  9. Generate a natural language response for explaining the comparison analytically understandable by financial users as well.
  10. Whenever you see any change or movement of values, provide an interpretation of the change in economic terms.
  11. If the result is empty or missing, say that no data was found for the requested query period.
  12. Return plain text only and do not format in bold font. Do not use Markdown formatting like **bold** or bullet points.

  Note that query and response details is provided in {output_json} and you must return a response.

  """
  # Initialize client with API key
  client = OpenAI(api_key=api_key)

  ## Call LLM model GPT-5.4 to generate query parser in JSON response. We arere using chat gpt model GPT-4o Realtime(gpt-4.5-preview).

  llm_response_generation = openai.chat.completions.create( model="gpt-5.4",
                  messages=[{"role":"system", "content":system_message},
                             {"role":"user","content":user_message}], temperature=0
                  )
  return llm_response_generation.choices[0].message.content


#### LLM Prompt 4: For generation of response to numeric and period wise comparision type of query.

In [111]:
## Creating system message and user message for prompt.

def numeric_period_compare_query_response_llm(output_json,api_key):
  system_message = """ You are an expert financial analysis assistant who is expertise in Macroeconomic analysis domain especially with FRED economic indcator and FOMC to generate human understandable response from user query. """

  user_message = f"""
  You are an expert financial analysis assistant for FOMC and FRED-based macroeconomic queries.
  You will be provided with structured query details and the computed numeric result in JSON format.
  Your task is to perform comparison analysis between two period range data of same indicator and generate a clear, human understandable, natural language comparative analysis response to the user's question using only from the provided structured input data.

  You must follow below guidelines before generating the response-
  1. Read and understand the user question from <<user_query>> parameter.
  2. Understand the user question context from <<type_of_comparision>> parameter.
  3. Read and perform comparison analysis of one indicators between different time range from below parameter values provided for both the period:
    a. <<indicator_label>>
    b. <<high_value>>
    c. <<low_value>>
    d. <<average_value>>
    e. <<first_value>>
    f. <<last_value>>
    g. <<slope_value>>
    h. <<period1_range>>
    i. <<trend>>
    j. <<volatility>>
  4. Read the value of parameter <<'period_having_higher_rate'>> to identify the duration when the indicator have higher rate value from the two diffferent time range period and use this information when generating a comparison summary.
  5. Parameter <<difference_of_rate_average>> will tell you the value of rate differences between two different time period of same indicator and use this information when generating a comparison summary.
  6. You must compare the rate value changes across two different time range of same indicator based on their trend, volatility, average value and slope.
  7. If for a specific time period the rate value have significantly changed, then mention the same by highlighting during the response generation.
  8. A chart will be required to visualise the trend and pattern of both time duration for the indicator and mention that a visual comparison chart is provided.
  9. Generate a natural language response for explaining the comparison analytically understandable by financial users as well.
  10. Whenever you see any change or movement of values, provide an interpretation of the change in economic terms.
  11. If the result is empty or missing, say that no data was found for the requested query period.
  12. Return plain text only and do not format in bold font. Do not use Markdown formatting like **bold** or bullet points.
  13. You need to give more importance in explaining how the indicator evolved/changed between the two time periods rather than repeating raw numbers.
  Note that query and response details is provided in {output_json} and you must return a response.

  """
  # Initialize client with API key
  client = OpenAI(api_key=api_key)

  ## Call LLM model GPT-5.4 to generate query parser in JSON response. We arere using chat gpt model GPT-4o Realtime(gpt-4.5-preview).

  llm_response_generation = openai.chat.completions.create( model="gpt-5.4",
                  messages=[{"role":"system", "content":system_message},
                             {"role":"user","content":user_message}], temperature=0
                  )
  return llm_response_generation.choices[0].message.content


#### LLM Prompt 5: For generation of response to summary type query based on topic/indicator.

In [112]:
## Creating system message and user message for prompt.

def text_summary_topic_query_response_llm(output_json,api_key):
  system_message = """ You are an expert financial analysis assistant who is expertise in Macroeconomic and policy analysis of FOMC textual data to generate human understandable response from user query. """

  user_message = f"""
  You are an expert financial analysis assistant for FOMC statements and minutes document textual query.
  You will be provided with structured FOMC text related query details in <<User_query>> and the optimized re-written query in <<Rewritten_query>> to obtain context of the query for generating summary based on indicator.
  You will be provided list of relevant chunks of meeting documents that are most relevant to user query in <<Retrieved_documents>>.
  Your task is to generate a comprehensive topic-based summary from retrieved Federal Open Market Committee (FOMC) meeting documents provided in <<Retrieved_documents>>.

  <<User_query>> : <<original_user_query>> parameter from <<output_json>>

  <<Rewritten_query>> : <<rewritten_user_query>> parameter from <<output_json>>

  <<Retrieved_documents>> : <<documents_retrieved_for_summary>> from <<output_json>>


  You must follow below instructions for generating the response:
  1. Identify the main topic requested by the user by analysing the user query in <<User_query>>.
  2. Review all the ten retrieved document excerpts from <<Retrieved_documents>> and identify statements most relevant to the topic.
  3. Extract below key and important economic insights to be included as part of summary.
    a. risk assessments
    b. meeting discussions
    c. economic indicators
    d. committee member participant's views and opinion
    e. policy stance
  4. If any overlapping insights or ideas across multiple documents then combine the information to produce a coherent explanation of the topic.
  5. Use only information provided in the retrieved documents and do not add any information of your own.
  6. You must not discuss indicators not related to topic or policy themes unless they are necessary to explain or provide context or support the explanation of requested topic.
  7. You must write as a professional and economic analyst.
  8. At the end of summary , you must provide citation of the documents that you used for generating summary.
  9. Follow below format for generating summary of the topic
  10. You must use '%' when refering to percentages and instead of writing the word 'percent' provide '%'.
      Example: 2% instead of 2 percent.

  Note that query and response details is provided in {output_json} and you must return a response.
  Output Format:

  Summary of topic:

  Write a concise analytical summary of 7–9 sentences explaining the topic discussed in FOMC documents.

  Key Insights:

  • Insight 1
  • Insight 2
  • Insight 3
  • Insight 4

  Citation Evidence:

  Mention the meeting dates referenced in the retrieved documents in format yyyy-mm-dd along with the source type as Minutes (M) or Statement(S) based on value provided in 'doc_type' parameter of <<documents_retrieved_for_summary>>.
  List all document citations using the STRICT format below:
  Document Date - YYYY-MM-DD , Source Type - Minutes
  Document Date - YYYY-MM-DD , Source Type - Statement
  """
  # Initialize client with API key
  client = OpenAI(api_key=api_key)

  ## Call LLM model GPT-5.4 to generate query parser in JSON response. We arere using chat gpt model GPT-4o Realtime(gpt-4.5-preview).

  llm_response_generation = openai.chat.completions.create( model="gpt-5.4",
                  messages=[{"role":"system", "content":system_message},
                             {"role":"user","content":user_message}], temperature=0
                  )
  return llm_response_generation.choices[0].message.content


#### LLM Prompt 6: For generation of response to summary type query based on FOMC Statement document type.

In [113]:
## Creating system message and user message for prompt.

def text_summary_fomc_statement_query_response_llm(output_json,api_key):
  system_message = """ You are an expert financial analysis assistant who is expertise in Macroeconomic and policy analysis of FOMC textual data to generate human understandable response from user query. """

  user_message = f"""
  You are an expert financial analysis assistant for FOMC statements only document textual query.
  You will be provided list of relevant chunks of FOMC statement documents for a specific period for generating comprehensive summary.
  Your task is to generate a comprehensive FOMC statement document summary from retrieved Federal Open Market Committee (FOMC) meeting documents provided in <<document_retreived_for_sumamry>>.

  <<document_retreived_for_sumamry>> from <<output_json>>

  You must follow below instructions for generating the summary response:
  1. Read and understand all the relevant documents excerpts from <<document_retreived_for_sumamry>>.
  2. Extract below key and important economic insights to be included as part of summary.
    a. risk assessments
    b. meeting discussions
    c. impact of economic indicators on policy
    d. committee member participant's views and opinion
    e. policy stance
  3. If any overlapping insights or ideas across multiple documents then combine the information to produce a coherent explanation of the topic.
  4. Use only information provided in the retrieved documents and do not add any information of your own.
  5. You must write as a professional and economic analyst.
  6. At the end of summary , you must provide citation of the documents that you used for generating summary.
  7. You must use '%' when refering to percentages and instead of writing the word 'percent' provide '%'.
      Example: 2% instead of 2 percent.

  Note that query type, document type, start date, emd date and relevant documents details are provided in {output_json} and you must return a response.
  <<start_date>> parameter value indicates the start date of FOMC statement.
  <<end_date>> parameter value indicates the end date of FOMC statement.

  Follow below format for generating summary of the topic
  Output Format:

  Summary of FOMC statement from <<start_date>> to <<end_date>>:

  Write a concise analytical summary of 7–9 sentences explaining the topic discussed in FOMC documents but do not exceed more than 200 words. Try to fit the first layer of summary in 200 words.

  Key Insights:

  • Insight 1
  • Insight 2
  • Insight 3
  • Insight 4

  Citation Evidence:

  Mention the meeting dates referenced in the retrieved documents in format yyyy-mm-dd and also provide citation link for each document.
  List all document citations using the STRICT format below:
  Document Date - YYYY-MM-DD , https://www.federalreserve.gov/newsevents/pressreleases/monetaryYYYYMMDDa.htm
  Document Date - YYYY-MM-DD , https://www.federalreserve.gov/newsevents/pressreleases/monetaryYYYYMMDDa.htm

  Replace YYYYMMDD in the link 'https://www.federalreserve.gov/newsevents/pressreleases/monetaryYYYYMMDDa.htm' with values present in document timestamp of document in <<document_retreived_for_sumamry>>.
  """
  # Initialize client with API key
  client = OpenAI(api_key=api_key)

  ## Call LLM model GPT-5.4 to generate query parser in JSON response. We arere using chat gpt model GPT-4o Realtime(gpt-4.5-preview).

  llm_response_generation = openai.chat.completions.create( model="gpt-5.4",
                  messages=[{"role":"system", "content":system_message},
                             {"role":"user","content":user_message}], temperature=0
                  )
  return llm_response_generation.choices[0].message.content


#### LLM Prompt 7: For generation of response to summary type query based on FOMC Minute document type.


In [114]:
## Creating system message and user message for prompt.

def text_summary_fomc_minute_query_response_llm(output_json,api_key):
  system_message = """ You are an expert financial analysis assistant who is expertise in Macroeconomic and policy analysis of FOMC textual data to generate human understandable response from user query. """

  user_message = f"""
  You are an expert financial analysis assistant for FOMC minutes only document textual query.
  You will be provided list of relevant chunks of FOMC minute documents for a specific period for generating comprehensive summary.
  Your task is to generate a comprehensive FOMC minute document summary from retrieved Federal Open Market Committee (FOMC) meeting documents provided in <<document_retreived_for_sumamry>>.

  <<document_retreived_for_sumamry>> from <<output_json>>

  You must follow below instructions for generating the summary response:
  1. Read and understand all the relevant documents excerpts from <<document_retreived_for_sumamry>>.
  2. Extract below key and important economic insights to be included as part of summary.
    a. risk assessments
    b. meeting discussions
    c. impact of economic indicators on policy
    d. committee member participant's views and opinion
    e. policy stance
  3. If any overlapping insights or ideas across multiple documents then combine the information to produce a coherent explanation of the topic.
  4. Use only information provided in the retrieved documents and do not add any information of your own.
  5. You must write as a professional and economic analyst.
  6. At the end of summary , you must provide citation of the documents that you used for generating summary.
  7. You must use '%' when refering to percentages and instead of writing the word 'percent' provide '%'.
      Example:3% instead of 3 percent.

  Note that query type, document type, start date, emd date and relevant documents details are provided in {output_json} and you must return a response.
  <<start_date>> parameter value indicates the start date of FOMC minute.
  <<end_date>> parameter value indicates the end date of FOMC minute.

  Follow below format for generating summary of the topic
  Output Format:

  Summary of FOMC minute from <<start_date>> to <<end_date>>:

  Write a concise analytical summary of 7–9 sentences explaining the topic discussed in FOMC minute documents but do not exceed more than 200 words. Try to fit the first layer of summary in 200 words.

  Key Insights:

  • Insight 1
  • Insight 2
  • Insight 3
  • Insight 4

  Citation Evidence:

  Mention the meeting dates referenced in the retrieved documents in format yyyy-mm-dd and also provide citation link for each document.
  List all document citations using the STRICT format below:
  Document Date - YYYY-MM-DD , https://www.federalreserve.gov/monetarypolicy/fomcminutesYYYYMMDD.htm
  Document Date - YYYY-MM-DD , https://www.federalreserve.gov/monetarypolicy/fomcminutesYYYYMMDD.htm

  Replace YYYYMMDD in the link 'https://www.federalreserve.gov/monetarypolicy/fomcminutesYYYYMMDD.htm' with values present in document timestamp of document in <<document_retreived_for_sumamry>>.
  """
  # Initialize client with API key
  client = OpenAI(api_key=api_key)

  ## Call LLM model GPT-5.4 to generate query parser in JSON response. We arere using chat gpt model GPT-4o Realtime(gpt-4.5-preview).

  llm_response_generation = openai.chat.completions.create( model="gpt-5.4",
                  messages=[{"role":"system", "content":system_message},
                             {"role":"user","content":user_message}], temperature=0
                  )
  return llm_response_generation.choices[0].message.content


#### LLM Prompt 8: For generation of response to question answer textual type query from FOMC documents.


In [115]:
## Creating system message and user message for prompt.

def text_question_answer_query_response_llm(output_json,api_key):
  system_message = """ You are an expert financial analysis assistant who is expertise in Macroeconomic and policy analysis of FOMC textual data to generate human understandable response from user query. """

  user_message = f"""
  You are an expert financial analysis assistant for FOMC minutes and statements document textual query.
  You will be provided list of relevant chunks of FOMC documents including both statement and minute type for a specific period.
  Your task is to generate answer to original user query related to FOMC communication documents from retrieved Federal Open Market Committee (FOMC) documents provided in <<documents_retrieved_for_question_answer>>.

  <<documents_retrieved_for_question_answer>> from <<output_json>>

  You must follow below instructions for generating the summary response:
  1. Read and understand all the relevant documents excerpts from <<documents_retrieved_for_question_answer>>.
  2. Read and understand the context of user original query from <<original_user_query>> parameter from <<output_json>>
  3. If any overlapping insights or ideas across multiple documents then combine the information to generate answer to user original query.
  4. Use only information provided in the retrieved documents and do not add any information of your own. The explanation in your answer must be simple, clear, concise, and factually grounded.
  5. You must not assume facts that are not present in the context and clearly mention in response if the answer generated from context document is partial.
  6. Whenever possible, mention the economic condition and policy stance.
  7. Write a direct answer to the user original question in 120–180 words and do not exceed more than 200 words. If required, you can use bullet points to explain the supporting information from documents.
  8. You must write as a professional and economic analyst.
  9. You must use '%' when refering to percentages and instead of writing the word 'percent' provide '%'.
      Example:3% instead of 3 percent.
  10. At the end of response, you must provide citation of the documents that you used for generating answer to user original query.

  Note that query type, original user query, meeting date and relevant documents details are provided in {output_json} and you must return a response.

  Citation Evidence:

  Mention the meeting dates referenced in the retrieved documents in format yyyy-mm-dd and also provide citation link for each document.
  You must ensure that for FOMC statement documents, the url link is 'https://www.federalreserve.gov/newsevents/pressreleases/monetaryYYYYMMDDa.htm'
  and for FOMC minute document url link is 'https://www.federalreserve.gov/monetarypolicy/fomcminutesYYYYMMDD.htm'
  You must use doicument type value based on 'doc_type' patrameter of <<documents_retrieved_for_question_answer>>.
  List all document citations using the STRICT format below:
  Document Date - YYYY-MM-DD , https://www.federalreserve.gov/monetarypolicy/fomcminutesYYYYMMDD.htm or https://www.federalreserve.gov/newsevents/pressreleases/monetaryYYYYMMDDa.htm based on document type.
  Document Date - YYYY-MM-DD , https://www.federalreserve.gov/monetarypolicy/fomcminutesYYYYMMDD.htm or https://www.federalreserve.gov/newsevents/pressreleases/monetaryYYYYMMDDa.htm based on document type.

  Replace YYYYMMDD in the url link  with values present in document timestamp of document in <<documents_retrieved_for_question_answer>>.

  """
  # Initialize client with API key
  client = OpenAI(api_key=api_key)

  ## Call LLM model GPT-5.4 to generate query parser in JSON response. We arere using chat gpt model GPT-4o Realtime(gpt-4.5-preview).

  llm_response_generation = openai.chat.completions.create( model="gpt-5.4",
                  messages=[{"role":"system", "content":system_message},
                             {"role":"user","content":user_message}], temperature=0
                  )
  return llm_response_generation.choices[0].message.content


# <font color = yellow><b> Chart Visualisation Agent :  To return plot of chart when user query demands for the plot of an indicator.

In [116]:
from overrides import final
from rich import print
import json, time
import matplotlib.dates as mdates

############################ Chart plot for trend of an indicator for timeseries type of query  #########################################

def chart_plot_of_trend_for_timeseries(data,summ):
  ## Set Date column as index.
  data.set_index('Date',inplace=True)
  ## To add trend in graph as liner straight line to understand the rise/fall in rate from 2000.
  x=np.arange(len(data.index))
  y=data['rate_value_%'].values
  coef=np.polyfit(x,y,1)    ## polyfit() function is to find best value of m(slope) and c(intercept) given value of x and y.
  trend=[]
  for i in x:
    trend.append(float(round(coef[0]*i+coef[1],2)))   ## first element in 'coef' is value of m(slope of line) and /
                                                    ##second element in 'coef' is value of c(intercept of linbe).
  ## Plot of trend
  plt.figure(figsize=(7,5))
  plt.plot(data.index, y, marker='o', linewidth=2.5, label='Actual')
  plt.plot(data.index,trend, linestyle='--', linewidth=2, label='Linear Trend')
  plt.title(f"{summ['indicator'][0]} trend/pattern in year {summ['start_date'][:4]}")
  plt.xlabel("Date (yyyy-mm-dd)")
  plt.ylabel(f"{summ['indicator'][0]} in %")
  plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
  plt.xticks(data.index,rotation=45)
  plt.grid(alpha=0.5)
  plt.legend()
  plt.show()

########################### Chart plot for trend of an indicator for comparision type of query ##########################################

def chart_plot_of_trend_for_comparision(data1,data2,summ):
  ## Plot for indicator_1
  ## Set Date column as index.
  data1.set_index('Date',inplace=True)
  ## To add trend in graph as liner straight line to understand the rise/fall in rate from 2000.
  x=np.arange(len(data1.index))
  y=data1['rate_value_%'].values
  coef=np.polyfit(x,y,1)    ## polyfit() function is to find best value of m(slope) and c(intercept) given value of x and y.
  trend=[]
  for i in x:
    trend.append(float(round(coef[0]*i+coef[1],2)))   ## first element in 'coef' is value of m(slope of line) and /
                                                    ##second element in 'coef' is value of c(intercept of linbe).
  ## Plot of trend
  plt.figure(figsize=(7,5))
  plt.plot(data1.index, y, marker='o', linewidth=2.5, label='Actual')
  plt.plot(data1.index,trend, linestyle='--', linewidth=2, label='Linear Trend')

  plt.title(f"{summ['indicator_1']['indicator_label']} trend/pattern in year {summ['indicator_1']['start_date'][:4]}")
  plt.xlabel("Date (yyyy-mm-dd)")
  plt.ylabel(f"{summ['indicator_1']['indicator_label']} in %")
  plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
  plt.xticks(data1.index,rotation=45)
  plt.grid(alpha=0.5)
  plt.legend()
  plt.show()
  print('\n\n')

  ## Plot for indicator_2
  ## Set Date column as index.
  data2.set_index('Date',inplace=True)
  ## To add trend in graph as liner straight line to understand the rise/fall in rate from 2000.
  x=np.arange(len(data2.index))
  y=data2['rate_value_%'].values
  coef=np.polyfit(x,y,1)    ## polyfit() function is to find best value of m(slope) and c(intercept) given value of x and y.
  trend=[]
  for i in x:
    trend.append(float(round(coef[0]*i+coef[1],2)))   ## first element in 'coef' is value of m(slope of line) and /
                                                    ##second element in 'coef' is value of c(intercept of linbe).
  ## Plot of trend
  plt.figure(figsize=(7,5))
  plt.plot(data2.index, y, marker='o', linewidth=2.5, label='Actual')
  plt.plot(data2.index,trend, linestyle='--', linewidth=2, label='Linear Trend')

  plt.title(f"{summ['indicator_2']['indicator_label']} trend/pattern in year {summ['indicator_2']['start_date'][:4]}")
  plt.xlabel("Date (yyyy-mm-dd)")
  plt.ylabel(f"{summ['indicator_2']['indicator_label']} in %")
  plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
  plt.xticks(data1.index,rotation=45)
  plt.grid(alpha=0.5)
  plt.legend()
  plt.show()

########################### Chart plot for trend of an indicator for comparision between two period ##########################################

def chart_plot_of_trend_for_period_comparision(data1,data2,summ):
  ## Plot for period_1
  ## Set Date column as index.
  data1.set_index('Date',inplace=True)
  ## To add trend in graph as liner straight line to understand the rise/fall in rate from 2000.
  x=np.arange(len(data1.index))
  y=data1['rate_value_%'].values
  coef=np.polyfit(x,y,1)    ## polyfit() function is to find best value of m(slope) and c(intercept) given value of x and y.
  trend=[]
  for i in x:
    trend.append(float(round(coef[0]*i+coef[1],2)))   ## first element in 'coef' is value of m(slope of line) and /
                                                    ##second element in 'coef' is value of c(intercept of linbe).
  ## Plot of trend
  plt.figure(figsize=(7,5))
  plt.plot(data1.index, y, marker='o', linewidth=2.5, label='Actual')
  plt.plot(data1.index,trend, linestyle='--', linewidth=2, label='Linear Trend')

  plt.title(f"{summ['period_1']['indicator_label'][0]} trend/pattern from {summ['period_1']['period1_range']}")
  plt.xlabel("Date (yyyy-mm-dd)")
  plt.ylabel(f"{summ['period_1']['indicator_label'][0]} in %")
  plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
  plt.grid(alpha=0.5)
  plt.gcf().autofmt_xdate()
  plt.legend()
  plt.show()
  print('\n\n')

  # Plot for period 2
  ## Set Date column as index.
  data2.set_index('Date',inplace=True)
  ## To add trend in graph as liner straight line to understand the rise/fall in rate from 2000.
  x=np.arange(len(data2.index))
  y=data2['rate_value_%'].values
  coef=np.polyfit(x,y,1)    ## polyfit() function is to find best value of m(slope) and c(intercept) given value of x and y.
  trend=[]
  for i in x:
    trend.append(float(round(coef[0]*i+coef[1],2)))   ## first element in 'coef' is value of m(slope of line) and /
                                                    ##second element in 'coef' is value of c(intercept of linbe).
  ## Plot of trend
  plt.figure(figsize=(7,5))
  plt.plot(data2.index, y, marker='o', linewidth=2.5, label='Actual')
  plt.plot(data2.index,trend, linestyle='--', linewidth=2, label='Linear Trend')
  plt.title(f"{summ['period_2']['indicator_label'][0]} trend/pattern from {summ['period_2']['period2_range']}")
  plt.xlabel("Date (yyyy-mm-dd)")
  plt.ylabel(f"{summ['period_2']['indicator_label'][0]} in %")
  plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
  plt.grid(alpha=0.5)
  plt.gcf().autofmt_xdate()
  plt.legend()
  plt.show()


### Initiate conversation with user for an interactive chatbot.

In [117]:
## user conversation chat bot function.

def initialize_conv():
    ## Start conversation with user.
    print('[bright_cyan]AI Assistant : Hello User, Please ask your question related to FOMC policy and FRED macroeconomic indicator. If you do not have any other query then please type "exit"[/bright_cyan]')
    retrieve_generate_time = [] ## To add time taken for model to retrieve and generate reply.
    while True:
      print('\n\nUser : ')
      user_query=input()
      if user_query.lower()=='exit':
        print('\n\n[bright_cyan]AI Assistant : Thank you for reaching us. Hope all your queries are resolved !![/bright_cyan]')
        break
      else:
        query_parse=query_intent_parser(user_query)
        strt=time.time()

        if query_parse['query_type']=='numeric' and query_parse['query_task_type']=='single':
          summ=query_orchestrator_for_retreival(query_parse,user_query)
          output_json=json.dumps(summ, indent=1)
          final_response=numeric_single_query_response_llm(output_json,api_key)

        elif query_parse['query_type']=='numeric' and query_parse['query_task_type']=='timeseries':
          summ,data=query_orchestrator_for_retreival(query_parse,user_query)
          output_json=json.dumps(summ, indent=1)
          final_response=numeric_timeseries_query_response_llm(output_json,api_key)

        elif query_parse['query_type']=='numeric' and query_parse['query_task_type']=='compare_indicator':
          summ,data1,data2=query_orchestrator_for_retreival(query_parse,user_query)
          output_json=json.dumps(summ, indent=1)
          final_response=numeric_indicator_compare_query_response_llm(output_json,api_key)

        elif query_parse['query_type']=='numeric' and query_parse['query_task_type']=='compare_period':
          summ,data1,data2=query_orchestrator_for_retreival(query_parse,user_query)
          output_json=json.dumps(summ, indent=1)
          final_response=numeric_period_compare_query_response_llm(output_json,api_key)

        elif query_parse['query_type']=='text' and query_parse['query_task_type']=='summary_topic':
          text_data=query_orchestrator_for_retreival(query_parse,user_query)
          output_json=json.dumps(text_data, indent=1)
          final_response=text_summary_topic_query_response_llm(output_json,api_key)

        elif query_parse['query_type']=='text' and query_parse['query_task_type']=='summary_fomc_statement':
          text_data=query_orchestrator_for_retreival(query_parse,user_query)
          output_json=json.dumps(text_data, indent=1)
          final_response=text_summary_fomc_statement_query_response_llm(output_json,api_key)

        elif query_parse['query_type']=='text' and query_parse['query_task_type']=='summary_fomc_minute':
          text_data=query_orchestrator_for_retreival(query_parse,user_query)
          output_json=json.dumps(text_data, indent=1)
          final_response=text_summary_fomc_minute_query_response_llm(output_json,api_key)

        elif query_parse['query_type']=='text' and query_parse['query_task_type']=='question_answer':
          text_data=query_orchestrator_for_retreival(query_parse,user_query)
          output_json=json.dumps(text_data, indent=1)
          final_response=text_question_answer_query_response_llm(output_json,api_key)

        end=time.time()
        # Calculate total time taken.
        retrieve_generate_time.append(round(end-strt,4))

        # Print response generated
        print(f'\n\n[bright_cyan]AI Assistant : [/bright_cyan]\n[bright_green]{final_response}[/bright_green]')

        # Display chart for timeseries and compare query type
        if query_parse['query_type']=='numeric' and query_parse['query_task_type']=='timeseries':
          print(f'\n\n[bright_green]Please find below plot of {summ['indicator'][0]} for visualisation :-[/bright_green]\n\n')
          chart_plot_of_trend_for_timeseries(data,summ)

        elif query_parse['query_type']=='numeric' and query_parse['query_task_type']=='compare_indicator':
          print(f'\n\n[bright_green]Please find below plot of {summ['indicator_1']['indicator_label']} and {summ['indicator_2']['indicator_label']} for visual comparision :-[/bright_green]\n\n')
          chart_plot_of_trend_for_comparision(data1,data2,summ)

        elif query_parse['query_type']=='numeric' and query_parse['query_task_type']=='compare_period':
          print(f'\n\n[bright_green]Please find below plot of {summ['period_1']['indicator_label'][0]} between two different period for visual comparision :-[/bright_green]\n\n')
          chart_plot_of_trend_for_period_comparision(data1,data2,summ)

        # Print time taken to generate answer to user
        print(f'\n\n[bright_cyan]Time Taken: {round(end-strt,4)} seconds[/bright_cyan]')
        print('\n\n[bright_cyan]AI Assistant : Please ask your next question related to Insurance Policies. If you do not have any other query then please type "exit"[/bright_cyan]')
    return retrieve_generate_time  ## Return individual time taken for questions.


In [119]:
time_taken=initialize_conv()  ## Call function to initiate chat conversation with user.



AI Assistant : Hello User, Please ask your question related to FOMC policy and FRED macroeconomic indicator. If you
do not have any other query then please type "exit"

User :

what was inflation rate in 2025?


AI Assistant : 
The Average Inflation Rate 5Year in 2025 was 2.41 %.

Time Taken: 0.8939 seconds

...

Time Taken: 28.7339 seconds

AI Assistant : Please ask your next question related to Insurance Policies. If you do not have any other query then
please type "exit"

User :

exit


AI Assistant : Thank you for reaching us. Hope all your queries are resolved !!